# MemoRAG extension: an 8B reader, an RWKV memory-size ladder, and a comparison

Extends [longmemeval_memorag.ipynb](longmemeval_memorag.ipynb) (MemoRAG with RWKV-7 G1 2.9B on
`Llama-3.2-1B-Instruct`) with three questions:

1. Does the MemoRAG effect hold with a stronger reader (`Llama-3.1-8B-Instruct`, served the same
   way `longmemeval_recreation.ipynb` already measured it -- OpenRouter/DeepInfra fp8)?
2. Does scaling the *memory module* (RWKV-7 G1 1.5B -> 2.9B -> 7.2B, same `20260805` release, same
   16,384-token trained context) change retrieval-clue quality?
3. How do all of these compare against `longmemeval_recreation.ipynb`'s own measured baseline?

**This notebook is a separate, standalone file, not a continuation of `longmemeval_memorag.ipynb`
that needs it run first.** A notebook cannot import from another (this repo's own convention --
see `paper_recreation/longmemeval_memorag.ipynb`'s own "fully self-contained" design and
`long_mem_eval_v1.ipynb`), so everything this notebook needs is transcribed here too, following
the same "transcribe, then prove the transcription with a gate" discipline. What it does NOT
re-derive is *work*: it mounts the identical Drive folder
(`/content/drive/MyDrive/lme_memorag/`) and reuses `longmemeval_memorag.ipynb`'s own cached RWKV
2.9B memory and Stella embeddings straight off disk if present -- no in-memory Python object is
shared, no upstream notebook needs to have run in this session, but a *prior* run of it on the
same Drive account means Stage A for the 2.9B rung and Stage B's embedding cache both start warm.

Everything this notebook writes lives under `MyDrive/lme_memorag/ext/` -- it never writes through
a path `longmemeval_memorag.ipynb` itself writes to. The two exceptions (both read-only) are its
cached RWKV 2.9B checkpoint/memory files and its Stella embedding cache, which are safe to reuse
because the corresponding writer functions here never touch those specific files unless building
that data from scratch is unavoidable (i.e. the base notebook never ran) -- and even then, new
work goes to `ext/store/`, never overwriting the original.

**Known deviations** (matching `longmemeval_memorag.ipynb`'s own Section 23 numbering, continued):

8. The RWKV ladder shares one recipe (`g1i`, `20260805`, `ctx16384`) but not every release size --
   a `g1j` (`20260831`) series also exists and is deliberately not mixed in.
9. New RWKV sizes get the `s` memory source only; `s_trunc16k` stays a 2.9B-only ablation.
10. The 8B reader is served fp8 by DeepInfra (matching `longmemeval_recreation.ipynb`'s own 8B
    row exactly); the 1B reader stays local bf16. Reader comparisons therefore confound scale with
    serving precision -- this is why the recreation notebook's own measured run, not the paper's
    published numbers, is the anchor for the 8B column in the final comparison section.
11. `longmemeval_recreation.ipynb` and `longmemeval_memorag.ipynb` sampled their 100-question
    subsets independently. This notebook reuses `longmemeval_memorag.ipynb`'s own subset
    (identical by construction: same `SEED`, same `PRESET`, same deterministic sampler) but that
    subset differs from the recreation notebook's. Every paired statistic against the recreation
    baseline is computed on the intersection, with the paired `n` printed; each notebook's own
    full-n number is also shown, unpaired, for context.

Runtime target: **under 6 hours** on a Colab A100-40GB, reusing every cached asset it can find.

## Section 1: Environment

In [ ]:
import os, sys, subprocess, json, math, time, hashlib, random, re, gc
from pathlib import Path
from datetime import datetime, timezone

if "google.colab" not in sys.modules:
    raise RuntimeError(
        "This notebook is Colab-only -- there is no local execution path. Open it in Google "
        "Colab (Runtime > Change runtime type > pick a GPU, ideally A100) and run it there.")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# SAME Drive folder longmemeval_memorag.ipynb uses -- this is how its cached RWKV-2.9B memory and
# Stella embeddings get reused without sharing any Python state with that notebook.
REPO = Path("/content/drive/MyDrive/lme_memorag")
REPO.mkdir(parents=True, exist_ok=True)

LOG_PATH = REPO / "logs" / "run_ext.log"   # separate log file -- keeps the two notebooks' logs
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data); s.flush()
    def flush(self):
        for s in self.streams:
            s.flush()
    def __getattr__(self, name):
        if name == "streams":
            raise AttributeError(name)
        return getattr(self.streams[0], name)
    def fileno(self):
        for s in self.streams:
            try:
                return s.fileno()
            except (AttributeError, OSError, ValueError):
                continue
        raise OSError("no underlying fileno available")

if not isinstance(sys.stdout, _Tee):
    _log_fh = open(LOG_PATH, "a", encoding="utf-8")
    _log_fh.write(f"\n{'='*80}\n[{datetime.now(timezone.utc).isoformat()}] session (re)started\n{'='*80}\n")
    _log_fh.flush()
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)

def detect_gpu():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None, 0.0
    if not out:
        return None, 0.0
    name, mib = out.splitlines()[0].split(",")
    return name.strip(), float(mib) / 1024

GPU_NAME, GPU_VRAM_GB = detect_gpu()
if GPU_NAME is None:
    raise RuntimeError("No GPU on this runtime. Runtime > Change runtime type > pick a GPU "
                       "(A100 recommended), then Runtime > Restart session and re-run from the top.")

CAN_RUN_LOCAL_READER = GPU_VRAM_GB >= 20     # Llama-3.2-1B via vLLM, generous headroom
CAN_RUN_RWKV         = GPU_VRAM_GB >= 16     # up to the 7.2B rung, fp16 (~14.4GB resident)
CAN_RUN_EMBEDDINGS   = GPU_VRAM_GB >= 8      # Stella V5 1.5B

print(f"Drive:      mounted, working folder {REPO}  (shared with longmemeval_memorag.ipynb)")
print(f"Log file:   {LOG_PATH}")
print(f"GPU:        {GPU_NAME}  ({GPU_VRAM_GB:.1f} GB)")
print(f"Local reader (Llama-3.2-1B via vLLM): {'yes' if CAN_RUN_LOCAL_READER else 'NO'}")
print(f"RWKV-7 G1 ladder (up to 7.2B):         {'yes' if CAN_RUN_RWKV else 'NO'}")
print(f"Stella V5 embeddings:                  {'yes' if CAN_RUN_EMBEDDINGS else 'NO'}")
if GPU_VRAM_GB < 38:
    print("\n  NOTE: designed and timed against an A100-40GB. This GPU has less VRAM -- Section "
          "10's per-model Gate B throughput estimate is the number to trust over any estimate in "
          "this notebook's own prose.")

## Section 2: Dependencies

In [ ]:
INSTALL = True   # set False on a re-run to skip

def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

if INSTALL:
    if "torch" in sys.modules:
        raise RuntimeError(
            "torch is already imported -- Runtime > Restart session and run from the top. A torch "
            "already in sys.modules shadows the one this cell installs.")

    if CAN_RUN_LOCAL_READER:
        pip_install("vllm==0.28.0", "torch==2.13.0+cu132", "torchvision==0.28.0+cu132",
                    "--index-url", "https://download.pytorch.org/whl/cu132",
                    "--extra-index-url", "https://pypi.org/simple")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)

    base = ["openai>=1.40", "tokenizers>=0.20", "numpy", "pandas", "matplotlib", "tqdm",
            "python-dotenv", "rank_bm25"]
    pip_install(*base)
    if CAN_RUN_RWKV:
        pip_install("rwkv")
    if CAN_RUN_EMBEDDINGS:
        pip_install("sentence-transformers")

import numpy as np, pandas as pd
print("dependencies ready")
if CAN_RUN_LOCAL_READER:
    import torch
    print(f"torch {torch.__version__}  (cuda {torch.version.cuda})")
    _v = subprocess.run([sys.executable, "-m", "pip", "show", "vllm"], capture_output=True, text=True).stdout
    print(next((l for l in _v.splitlines() if l.startswith("Version:")), "vllm version: unknown"))

## Section 3: Configuration

In [ ]:
# Must match longmemeval_memorag.ipynb's own PRESET/SEED to reuse its cached 100-question subset
# and RWKV-2.9B memory rows -- choose_questions() (Section 9) is deterministic given (SEED, n), so
# matching values here guarantee an IDENTICAL sample, not just a same-size one. This is checked,
# not just assumed -- Section 9 cross-checks against the base notebook's own cached subset file
# when present.
PRESET = "pilot"          # "smoke" (5) | "pilot" (100) | "full" (500)
N_QUESTIONS = {"smoke": 5, "pilot": 100, "full": 500}[PRESET]
SEED = 0

# --- readers -------------------------------------------------------------------------------------
READERS = {
    "llama-3.2-1b-instruct": dict(
        route="local", hf_id="meta-llama/Llama-3.2-1B-Instruct",
        native_context=128000, served_context=128000,
        tokenizer_stem="llama-3.2-3b-instruct", tokenizer_hf_id="meta-llama/Llama-3.2-3B-Instruct",
        note="Shares its tokenizer with the 3B model (same family). Runs local -- OpenRouter's "
             "only provider caps at 60k tokens, well under what an S question needs."),
    "llama-3.1-8b-instruct": dict(
        route="openrouter", or_id="meta-llama/llama-3.1-8b-instruct", provider="DeepInfra",
        native_context=128000, served_context=131072,
        tokenizer_stem="llama-3.1-8b-instruct", tokenizer_hf_id="meta-llama/Llama-3.1-8B-Instruct",
        note="DeepInfra serves fp8 at 131k -- matches longmemeval_recreation.ipynb's own 8B row "
             "exactly, so the final comparison isn't confounded by a precision change (deviation #10)."),
}
READER_ORDER = ["llama-3.2-1b-instruct", "llama-3.1-8b-instruct"]

JUDGE_MODEL = "gpt-4o-2024-08-06"        # the paper's own judge, OpenAI API
TEMPERATURE = 0.0                        # greedy decoding throughout, matching both papers

# --- RWKV memory-model ladder -------------------------------------------------------------------
# Same family (BlinkDL/rwkv7-g1), same release (20260805), same trained context (16384) --
# head_dim=64 confirmed live against each size's HF config.json; checked again per-model at load
# time (Section 10's Gate E3) since the `rwkv` package's fused CUDA kernel is compiled with a
# hardcoded HEAD_SIZE=64 -- a mismatch would silently misread tensor layout rather than error.
RWKV_HF_REPO = "BlinkDL/rwkv7-g1"
RWKV_LADDER = {
    "g1i-1.5b":  "rwkv7-g1i-1.5b-20260805-ctx16384",
    "g1i-2.9b":  "rwkv7-g1i-2.9b-20260805-ctx16384",   # same weights longmemeval_memorag.ipynb uses
    "g1i-7.2b":  "rwkv7-g1i-7.2b-20260805-ctx16384",
    "g1i-13.3b": "rwkv7-g1i-13.3b-20260805-ctx16384",   # NOT active by default -- 26.5GB / ~2.6h
}
RWKV_ACTIVE = ["g1i-1.5b", "g1i-2.9b", "g1i-7.2b"]
RWKV_PARAMS_B = {"g1i-1.5b": 1.5, "g1i-2.9b": 2.9, "g1i-7.2b": 7.2, "g1i-13.3b": 13.3}
STAGE_A_BUDGET_H = {"g1i-1.5b": 0.75, "g1i-2.9b": 1.25, "g1i-7.2b": 2.5, "g1i-13.3b": 4.5}
RWKV_STRATEGY = "cuda fp16"
CHUNK_TOKENS = 1024
NOTE_MAX_TOKENS = 128
RWKV_CUDA_KERNEL = True
RWKV_TRUNCATE_TOKENS = 16384

# New-tag checkpoints go to LOCAL disk, not Drive: 1.5B+7.2B = 17.5GB the base notebook's own
# 2.9B checkpoint didn't need, and they're cheaply re-downloadable -- only the built memory JSONL
# rows (a few KB each) need to survive a disconnected runtime.
_RWKV_LOCAL_DIR = Path("/content/rwkv_ckpt")
_RWKV_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# --- retrieval --------------------------------------------------------------------------------
STELLA_HF = "NovaSearch/stella_en_1.5B_v5"
PRIMARY_DENSE_RETRIEVER = "stella"
RETRIEVERS = ["stella", "bm25"]
QUERY_DESIGN = "sur+span"   # matches longmemeval_memorag.ipynb's own memorag_top{5,10} design

OR_CONCURRENCY = 16

# --- paths -----------------------------------------------------------------------------------
# DATA_DIR / STORE are SHARED with longmemeval_memorag.ipynb (same Drive REPO) -- dataset files,
# the base notebook's RWKV-2.9B checkpoint/tokenizers, and (crucially) the sha1-keyed Stella
# embedding cache all live here and are safely reused/extended, never overwritten, by this
# notebook. RUNS / RESULTS / MEM_STORE are exclusive to this notebook, under ext/, so nothing here
# can ever collide with or corrupt a file longmemeval_memorag.ipynb itself writes.
DATA_DIR = REPO / "data"
STORE = REPO / "store"
MEM_STORE, RUNS, RESULTS = REPO / "ext" / "store", REPO / "ext" / "runs", REPO / "ext" / "results"
for _d in (DATA_DIR, STORE, MEM_STORE, RUNS, RESULTS):
    _d.mkdir(parents=True, exist_ok=True)
del _d

RECREATION_REPO = Path("/content/drive/MyDrive/lme_retry")   # longmemeval_recreation.ipynb's own
                                                                # root -- read-only from here.

random.seed(SEED); np.random.seed(SEED)

print(f"PRESET={PRESET}  ->  n={N_QUESTIONS} questions")
print('readers: ' + ', '.join(f"{r} [{READERS[r]['route']}]" for r in READER_ORDER))
print(f"RWKV ladder (active): {', '.join(RWKV_ACTIVE)}  (13.3b wired up, off by default)")
print(f"shared data:   {DATA_DIR}  /  {STORE}")
print(f"this notebook: {RUNS.parent}  (RUNS / RESULTS / store under ext/)")
print(f"recreation baseline (read-only): {RECREATION_REPO}  "
      f"({'found' if RECREATION_REPO.exists() else 'NOT FOUND -- final comparison will be limited'})")

## Section 4: API keys

In [ ]:
def load_env():
    candidates = [REPO / ".env", Path("/content/drive/MyDrive/lme_memorag/.env")]
    for env_path in candidates:
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip())
            return env_path
    return None

SECRET_NAMES = ("OPENAI_API_KEY", "OPENROUTER_API_KEY", "HF_TOKEN")

env_loaded_from = load_env()
from google.colab import userdata
for k in SECRET_NAMES:
    if not os.environ.get(k):
        try:
            os.environ[k] = userdata.get(k)
        except Exception as e:
            print(f"  [warn] Colab secret {k} unavailable: {e}")

missing = [k for k in SECRET_NAMES if not os.environ.get(k)]
if missing:
    raise RuntimeError(
        f"Missing {', '.join(missing)}. Either: (1) click the key icon in the left Colab sidebar, "
        f"add secrets named {', '.join(missing)}, and toggle on notebook access for each; or "
        f"(2) put them as KEY=value lines in a .env file at {REPO}/.env on your Drive, which "
        f"persists across sessions so you never upload or retype them again."
    )

from openai import OpenAI
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
openrouter_client = OpenAI(api_key=os.environ.get("OPENROUTER_API_KEY"),
                           base_url="https://openrouter.ai/api/v1")
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_TOKEN", ""))

print("OpenAI key:     ", "set" if os.environ.get("OPENAI_API_KEY") else "MISSING")
print("OpenRouter key: ", "set" if os.environ.get("OPENROUTER_API_KEY") else "MISSING")
print("HF token:       ", "set" if os.environ.get("HF_TOKEN") else "MISSING")
print("env file used:  ", env_loaded_from or "(none found -- used Colab secrets)")

## Section 5: The dataset, and Gate 1

In [ ]:
from collections import Counter

EXPECTED_QTYPES = {
    "temporal-reasoning": 133, "multi-session": 133, "knowledge-update": 78,
    "single-session-user": 70, "single-session-assistant": 56, "single-session-preference": 30,
}
DATASET_BASE_URL = "https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main"


def download_if_absent(name):
    import urllib.request
    dest = DATA_DIR / name
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{DATASET_BASE_URL}/{name}"
    print(f"  {name}: downloading from {url} ...")
    t0 = time.time()
    tmp = dest.with_suffix(dest.suffix + ".part")
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f"    -> {dest}  ({dest.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.0f}s)")
    return dest


def load_dataset(name):
    with open(download_if_absent(name), encoding="utf-8") as f:
        return json.load(f)

ORACLE = load_dataset("longmemeval_oracle.json")
S_DATA = load_dataset("longmemeval_s_cleaned.json")

def gate1_dataset_integrity():
    ok = True
    for label, data in [("oracle", ORACLE), ("S", S_DATA)]:
        n = len(data)
        good = n == 500
        ok &= good
        print(f"  [{'OK' if good else 'FAIL'}] {label}: {n} questions (expected 500)")
        counts = dict(Counter(q["question_type"] for q in data))
        good = counts == EXPECTED_QTYPES
        ok &= good
        print(f"  [{'OK' if good else 'FAIL'}] {label}: question-type counts match Figure 9(a)")
        if not good:
            print(f"        got {counts}")
    same_ids = {q["question_id"] for q in ORACLE} == {q["question_id"] for q in S_DATA}
    ok &= same_ids
    print(f"  [{'OK' if same_ids else 'FAIL'}] oracle and S reference the same 500 question ids")
    print(f"\nGate 1: {'PASSED' if ok else 'FAILED'}")
    return ok

gate1_dataset_integrity()
QID2ORACLE = {q["question_id"]: q for q in ORACLE}
QID2S = {q["question_id"]: q for q in S_DATA}

## Section 6: The method, transcribed from the authors' code and from MemoRAG

Identical to `longmemeval_memorag.ipynb`'s own Sections 6/8 -- transcribed independently here
because a notebook cannot import from another. Gates 2a/2b below prove both transcriptions still
match live upstream source.

In [ ]:
# --- Reader prompt templates, verbatim from run_generation.py's prepare_prompt() -----------------
PROMPT_TEMPLATES = {
    ("none", False):
        "I will give you several history chats between you and a user. Please answer the question based on the relevant chat history.\n\n\nHistory Chats:\n\n{}\n\nCurrent Date: {}\nQuestion: {}\nAnswer:",
    ("none", True):
        "I will give you several history chats between you and a user. Please answer the question based on the relevant chat history. Answer the question step by step: first extract all the relevant information, and then reason over the information to get the answer.\n\n\nHistory Chats:\n\n{}\n\nCurrent Date: {}\nQuestion: {}\nAnswer (step by step):",
    ("merge", False):
        "I will give you several history chats between you and a user, as well as the relevant user facts extracted from the chat history. Please answer the question based on the relevant chat history and the user facts\n\n\nHistory Chats:\n\n{}\n\nCurrent Date: {}\nQuestion: {}\nAnswer:",
    ("merge", True):
        "I will give you several history chats between you and a user, as well as the relevant user facts extracted from the chat history. Please answer the question based on the relevant chat history and the user facts. Answer the question step by step: first extract all the relevant information, and then reason over the information to get the answer.\n\n\nHistory Chats:\n\n{}\n\nCurrent Date: {}\nQuestion: {}\nAnswer (step by step):",
    ("replace", False):
        "I will give you several facts extracted from history chats between you and a user. Please answer the question based on the relevant facts.\n\n\nHistory Chats:\n\n{}\n\nCurrent Date: {}\nQuestion: {}\nAnswer:",
    ("replace", True):
        "I will give you several facts extracted from history chats between you and a user. Please answer the question based on the relevant facts. Answer the question step by step: first extract all the relevant information, and then reason over the information to get the answer.\n\n\nHistory Chats:\n\n{}\n\nCurrent Date: {}\nQuestion: {}\nAnswer (step by step):",
}
PROMPT_TEMPLATE_CLOSEDBOOK = (
    "Answer the question based on your own knowledge. If you don't know the answer, say so.\n\n"
    "Current Date: {}\nQuestion: {}\nAnswer:")

GEN_LENGTH_BY_READING = {"direct": 500, "con": 800}
TEMPERATURE = 0.0

def max_retrieval_length(reader, reading):
    ctx = min(READERS[reader]["native_context"], READERS[reader]["served_context"])
    return ctx - GEN_LENGTH_BY_READING[reading] - 1000

def strip_has_answer(turns):
    return [{k: v for k, v in t.items() if k != "has_answer"} if isinstance(t, dict) else t
            for t in turns]

def render_chunks(chunks, history_format, merge_mode="none", granularity="session"):
    chunks = sorted(chunks, key=lambda x: x[0])
    out = ""
    for i, item in enumerate(chunks):
        if merge_mode == "merge":
            date, expansion, entry = item
        else:
            date, entry = item
            expansion = None
        if history_format == "json":
            if merge_mode == "merge":
                sess = "\n" + json.dumps({"session_summary_facts": expansion, "original_session": entry})
            else:
                sess = "\n" + json.dumps(entry)
        elif history_format == "nl":
            sess = ""
            if merge_mode == "merge":
                sess += "\n\nSession summary and facts:" + str(expansion)
            if isinstance(entry, list):
                for turn in entry:
                    sess += "\n\n{}: {}".format(turn["role"], turn["content"].strip())
            else:
                sess += "{}: {}".format(entry["role"], entry["content"].strip())
        else:
            raise NotImplementedError(history_format)
        out += "\n### Session {}:\nSession Date: {}\nSession Content:\n{}\n".format(i + 1, date, sess)
    return out

def build_prompt(question_entry, chunks, reader, reading, history_format="json",
                 merge_mode="none", granularity="session"):
    history = render_chunks(chunks, history_format, merge_mode, granularity)
    assert history != "", "empty history string -- run_generation.py asserts the same"
    history = truncate_history(history, reader, reading)
    template = PROMPT_TEMPLATES[(merge_mode, reading == "con")]
    return template.format(history, question_entry["question_date"], question_entry["question"])

# --- Judge prompts, verbatim from evaluate_qa.py's get_anscheck_prompt() -------------------------
JUDGE_TEMPLATE_STANDARD = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
JUDGE_TEMPLATE_TEMPORAL = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. In addition, do not penalize off-by-one errors for the number of days. If the question asks for the number of days/weeks/months, etc., and the model makes off-by-one errors (e.g., predicting 19 days when the answer is 18), the model's response is still correct. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
JUDGE_TEMPLATE_KNOWLEDGE_UPDATE = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response contains some previous information along with an updated answer, the response should be considered as correct as long as the updated answer is the required answer.\n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
JUDGE_TEMPLATE_PREFERENCE = "I will give you a question, a rubric for desired personalized response, and a response from a model. Please answer yes if the response satisfies the desired response. Otherwise, answer no. The model does not need to reflect all the points in the rubric. The response is correct as long as it recalls and utilizes the user's personal information correctly.\n\nQuestion: {}\n\nRubric: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
JUDGE_TEMPLATE_ABSTENTION = "I will give you an unanswerable question, an explanation, and a response from a model. Please answer yes if the model correctly identifies the question as unanswerable. The model could say that the information is incomplete, or some other information is given but the asked information is not.\n\nQuestion: {}\n\nExplanation: {}\n\nModel Response: {}\n\nDoes the model correctly identify the question as unanswerable? Answer yes or no only."

def judge_prompt(question_type, question, answer, response, abstention):
    if abstention:
        t = JUDGE_TEMPLATE_ABSTENTION
    elif question_type in ("single-session-user", "single-session-assistant", "multi-session"):
        t = JUDGE_TEMPLATE_STANDARD
    elif question_type == "temporal-reasoning":
        t = JUDGE_TEMPLATE_TEMPORAL
    elif question_type == "knowledge-update":
        t = JUDGE_TEMPLATE_KNOWLEDGE_UPDATE
    elif question_type == "single-session-preference":
        t = JUDGE_TEMPLATE_PREFERENCE
    else:
        raise NotImplementedError(question_type)
    return t.format(question, answer, response)

# --- corpus construction + retrieval metrics ------------------------------------------------------
def build_corpus(q, granularity):
    ids, values, keys, stamps = [], [], [], []
    for date, sid, turns in zip(q["haystack_dates"], q["haystack_session_ids"], q["haystack_sessions"]):
        clean = strip_has_answer(turns)
        if granularity == "session":
            ids.append(sid); values.append(clean); stamps.append(date)
            keys.append(" ".join(t["content"] for t in clean if t.get("role") == "user"))
        else:
            for i, turn in enumerate(turns):
                if turn["role"] != "user":
                    continue
                rnd = clean[i:i + 2]
                ids.append(f"{sid}_{i + 1}"); values.append(rnd); stamps.append(date)
                keys.append(turn.get("content", ""))
    return ids, values, keys, stamps

def evidence_ids(q, granularity):
    if granularity == "session":
        return set(q["answer_session_ids"])
    out = set()
    for sid, turns in zip(q["haystack_session_ids"], q["haystack_sessions"]):
        for i, turn in enumerate(turns):
            if turn.get("has_answer"):
                out.add(f"{sid}_{i + 1}")
    return out

def dcg(relevances, k):
    rel = np.asarray(relevances, dtype=float)[:k]
    if rel.size:
        return rel[0] + np.sum(rel[1:] / np.log2(np.arange(2, rel.size + 1)))
    return 0.0

def ndcg_any(rankings, correct, corpus_ids, k=10):
    rel = [1 if cid in correct else 0 for cid in corpus_ids]
    actual = dcg([rel[i] for i in rankings[:k]], k)
    ideal = dcg(sorted(rel, reverse=True), k)
    return 0.0 if ideal == 0 else actual / ideal

def evaluate_retrieval(rankings, correct, corpus_ids, k=10):
    recalled = set(corpus_ids[i] for i in rankings[:k])
    recall_any = float(any(d in recalled for d in correct))
    recall_all = float(all(d in recalled for d in correct)) if correct else 0.0
    return recall_any, recall_all, ndcg_any(rankings, correct, corpus_ids, k)

print("Transcribed: 6 reader templates + closedbook, 5 judge templates, corpus builder, retrieval metrics.")

## Section 7: Gate 2a -- proving the LongMemEval transcription still matches the live source

In [ ]:
import ast, urllib.request

LME_REPO_BASE = "https://raw.githubusercontent.com/xiaowu0162/LongMemEval/main"
_source_cache = {}

def fetch_source(url_or_path, base=LME_REPO_BASE):
    key = (base, url_or_path)
    if key in _source_cache:
        return _source_cache[key]
    try:
        with urllib.request.urlopen(f"{base}/{url_or_path}", timeout=30) as r:
            src = r.read().decode("utf-8")
    except Exception as e:
        print(f"  WARNING: could not fetch {url_or_path} ({e}) -- skipping its checks (warn, not fail).")
        src = None
    _source_cache[key] = src
    return src

def string_literals(src):
    return {n.value for n in ast.walk(ast.parse(src))
            if isinstance(n, ast.Constant) and isinstance(n.value, str)}

_GATE2_TOTALS = {"checked": 0, "passed": 0}

def gate2_check(label, ok):
    _GATE2_TOTALS["checked"] += 1
    _GATE2_TOTALS["passed"] += bool(ok)
    print(f"  [{'OK' if ok else 'MISMATCH'}] {label}")

def gate2a_longmemeval_fidelity():
    gen = fetch_source("src/generation/run_generation.py")
    if gen:
        lits = string_literals(gen)
        for key, tmpl in PROMPT_TEMPLATES.items():
            gate2_check(f"reader template {key} is a literal in run_generation.py", tmpl in lits)

    ev = fetch_source("src/evaluation/evaluate_qa.py")
    if ev:
        lits = string_literals(ev)
        for name, tmpl in [("STANDARD", JUDGE_TEMPLATE_STANDARD), ("TEMPORAL", JUDGE_TEMPLATE_TEMPORAL),
                           ("KNOWLEDGE_UPDATE", JUDGE_TEMPLATE_KNOWLEDGE_UPDATE),
                           ("PREFERENCE", JUDGE_TEMPLATE_PREFERENCE),
                           ("ABSTENTION", JUDGE_TEMPLATE_ABSTENTION)]:
            gate2_check(f"judge template {name} is a literal in evaluate_qa.py", tmpl in lits)

    print(f"\nGate 2a: {_GATE2_TOTALS['passed']}/{_GATE2_TOTALS['checked']} checks passed against "
          f"the LongMemEval authors' live source.")
    if _GATE2_TOTALS["checked"] and _GATE2_TOTALS["passed"] < _GATE2_TOTALS["checked"]:
        print("  A transcribed string no longer matches upstream -- re-check before trusting results.")

gate2a_longmemeval_fidelity()

## Section 8: MemoRAG prompts, and Gate 2b -- proving the MemoRAG transcription still matches the live source

In [ ]:
MEMORAG_REPO_BASE = "https://raw.githubusercontent.com/qhjqhj00/MemoRAG/main"

# Verbatim from memorag/prompt.py's en_prompts dict.
MEMORAG_PROMPTS = {
    "sur": (
        "\nYou are given a question related to the article. To answer it effectively, you need to "
        "recall specific details from the article. Your task is to generate precise clue questions "
        "that can help locate the necessary information.\n\n### Question: {question}\n### "
        "Instructions:\n1. You have a general understanding of the article. Your task is to "
        "generate one or more specific clues that will help in searching for supporting evidence "
        "within the article.\n2. The clues are in the form of precise surrogate questions that "
        "clarify the original question.\n3. Only output the clues. If there are multiple clues, "
        "separate them with a newline."
    ),
    "span": (
        "\nYou are given a question related to the article. To answer it effectively, you need to "
        "recall specific details from the article. Your task is to identify and extract one or "
        "more specific clue texts from the article that are relevant to the question.\n\n### "
        "Question: {question}\n### Instructions:\n1. You have a general understanding of the "
        "article. Your task is to generate one or more specific clues that will help in searching "
        "for supporting evidence within the article.\n2. The clues are in the form of text spans "
        "that will assist in answering the question.\n3. Only output the clues. If there are "
        "multiple clues, separate them with a newline."
    ),
    "qa": (
        "\nYou are given a question related to the article. Your task is to answer the question "
        "directly.\n\n### Question: {question}\n### Instructions:\nProvide a direct answer to the "
        "question based on the article's content. Do not include any additional text beyond the "
        "answer."
    ),
}

def gate2b_memorag_fidelity():
    prompt_src = fetch_source("memorag/prompt.py", base=MEMORAG_REPO_BASE)
    if prompt_src:
        lits = string_literals(prompt_src)
        for kind, tmpl in MEMORAG_PROMPTS.items():
            gate2_check(f"MemoRAG en_prompts['{kind}'] is a literal in memorag/prompt.py", tmpl in lits)

    mem_src = fetch_source("memorag/memorag.py", base=MEMORAG_REPO_BASE)
    if mem_src:
        tree = ast.parse(mem_src)
        found = {"filter_gt3": False, "append_query_last": False}
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef) and node.name == "_prepare_retrieval_query":
                src_of_fn = ast.get_source_segment(mem_src, node) or ""
                found["filter_gt3"] = "len(q.split()) > 3" in src_of_fn
                calls = re.findall(r"retrieval_query\.append\((\w+)\)", src_of_fn)
                found["append_query_last"] = bool(calls) and calls[-1] == "query"
        gate2_check("_prepare_retrieval_query still filters candidates to >3 words", found["filter_gt3"])
        gate2_check("_prepare_retrieval_query still appends the raw query last", found["append_query_last"])

    print(f"\nGate 2b: {_GATE2_TOTALS['passed']}/{_GATE2_TOTALS['checked']} checks passed so far "
          f"(cumulative with Gate 2a) against MemoRAG's live source.")
    if _GATE2_TOTALS["checked"] and _GATE2_TOTALS["passed"] < _GATE2_TOTALS["checked"]:
        print("  A transcribed string or behavior no longer matches upstream -- re-check before "
              "trusting any MemoRAG-side result below.")

gate2b_memorag_fidelity()

## Section 9: Tokenizers, and the question subset

Reuses `longmemeval_memorag.ipynb`'s own cached `question_subset_{n}.json` if present (same
`STORE`) -- and even if it's missing, `choose_questions()` is the identical, deterministic
function, so a from-scratch sample at the same `(SEED, N_QUESTIONS)` is guaranteed identical.

In [ ]:
from tokenizers import Tokenizer

TOKENIZER_DIR = DATA_DIR / "tokenizers"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
_tokenizers = {}

def get_tokenizer(reader):
    if reader in _tokenizers:
        return _tokenizers[reader]
    spec = READERS[reader]
    path = TOKENIZER_DIR / f"{spec['tokenizer_stem']}.json"
    if path.exists():
        tk = Tokenizer.from_file(str(path))
    else:
        tk = Tokenizer.from_pretrained(spec["tokenizer_hf_id"])   # gated repos need HF_TOKEN set
        tk.save(str(path))
        print(f"  cached tokenizer -> {path}")
    _tokenizers[reader] = tk
    return tk

def truncate_history(history, reader, reading):
    limit = max_retrieval_length(reader, reading)
    tk = get_tokenizer(reader)
    ids = tk.encode(history, add_special_tokens=False).ids
    if len(ids) <= limit:
        return history
    return tk.decode(ids[:limit], skip_special_tokens=True)

def count_tokens(text, reader):
    return len(get_tokenizer(reader).encode(text, add_special_tokens=False).ids)

for r in READER_ORDER:
    print(f"  {r:24s} tokenizer ok, max_retrieval_length(direct)={max_retrieval_length(r,'direct'):>7,}"
          f"  (con)={max_retrieval_length(r,'con'):>7,}")


SUBSET_PATH = STORE / f"question_subset_{N_QUESTIONS}.json"   # SAME path longmemeval_memorag.ipynb
                                                                 # writes/reads -- shared STORE.

def choose_questions(n):
    if n >= 500:
        return [q["question_id"] for q in S_DATA]
    if SUBSET_PATH.exists():
        ids = json.loads(SUBSET_PATH.read_text())
        if len(ids) == n:
            print(f"  reusing the saved {n}-question subset (shared with longmemeval_memorag.ipynb)")
            return ids
    by_type = {}
    for q in S_DATA:
        by_type.setdefault(q["question_type"], []).append(q["question_id"])
    rng = random.Random(SEED)
    picked = []
    for t, qs in sorted(by_type.items()):
        share = max(1, round(n * len(qs) / len(S_DATA)))
        picked += rng.sample(qs, min(share, len(qs)))
    picked = picked[:n]
    SUBSET_PATH.write_text(json.dumps(picked))
    return picked

QUESTION_IDS = choose_questions(N_QUESTIONS)
print(f"  {len(QUESTION_IDS)} questions selected")
print("  by type:", dict(Counter(QID2S[i]["question_type"] for i in QUESTION_IDS)))

## Section 10: Stage A -- building RWKV memory across the size ladder

Builds the `s` memory source (feed the full haystack, probe `sur`/`span`/`qa`) for every tag in
`RWKV_ACTIVE`, plus the `s_trunc16k` ablation for `g1i-2.9b` only (deviation #9). For `g1i-2.9b`,
`memory_path()` reuses `longmemeval_memorag.ipynb`'s own cached files in place if present (read
-only) -- if that notebook never ran, it transparently falls back to building `g1i-2.9b` fresh
here, so this notebook works standalone either way.

Gate A (correctness) runs once, against whichever tag is loaded first. Gate B (throughput) runs
per tag and hard-asserts against `STAGE_A_BUDGET_H` before that tag's full 100-question loop, so a
slower-than-expected model stops the run rather than quietly consuming the session.

In [ ]:
import glob, shutil, urllib.request

_RWKV_DIR_BASE = DATA_DIR / "rwkv"   # shared with longmemeval_memorag.ipynb -- g1i-2.9b's
_RWKV_DIR_BASE.mkdir(parents=True, exist_ok=True)   # checkpoint lives here if it downloaded it.

def _rwkv_checkpoint_dir(tag):
    # g1i-2.9b's checkpoint goes to the SAME shared location the base notebook uses (so it's
    # reused, not re-downloaded, if that notebook already fetched it); every other tag downloads
    # to local Colab disk (ephemeral, cheaply re-downloadable -- avoids a 17.5GB Drive footprint
    # the base notebook never needed).
    return _RWKV_DIR_BASE if tag == "g1i-2.9b" else _RWKV_LOCAL_DIR

def _download_url_if_absent(url, dest_dir):
    dest = dest_dir / Path(url).name
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  {dest.name}: already present ({dest.stat().st_size / 1e6:.1f} MB), skipping download.")
        return dest
    print(f"  {dest.name}: downloading from {url} ...")
    t0 = time.time()
    tmp = dest.with_suffix(dest.suffix + ".part")
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f"    -> {dest}  ({dest.stat().st_size / 1e6:.0f} MB in {time.time() - t0:.0f}s)")
    return dest


def load_rwkv(tag):
    # Rebinds the module-level rwkv_model / rwkv_pipeline globals that feed() and probe_memory()
    # close over -- one model resident at a time, matching longmemeval_memorag.ipynb's own pattern.
    global rwkv_model, rwkv_pipeline
    model_file = RWKV_LADDER[tag]
    url = f"https://huggingface.co/{RWKV_HF_REPO}/resolve/main/{model_file}.pth"
    dest = _download_url_if_absent(url, _rwkv_checkpoint_dir(tag))

    # Gate E3 -- kernel safety. The rwkv package's fused CUDA kernel is compiled once, process-wide,
    # with a hardcoded HEAD_SIZE=64 macro. Every g1i checkpoint is confirmed head_dim=64 via its
    # own HF config.json, but this is checked again here, live, against the actual downloaded
    # weights -- a mismatch would silently misinterpret tensor layout rather than raise.
    try:
        raw = __import__("torch").load(dest, map_location="cpu", mmap=True)
    except Exception:
        raw = __import__("torch").load(dest, map_location="cpu")
    n_head, head_size = raw["blocks.0.att.r_k"].shape
    del raw
    assert head_size == 64, (
        f"{tag}: head_size={head_size}, expected 64 -- the rwkv package's fused CUDA kernel is "
        f"compiled with a hardcoded HEAD_SIZE=64. Do not run this tag without matching HEAD_SIZE "
        f"or disabling RWKV_CUDA_ON for it.")
    print(f"  {tag}: Gate E3 head_size=64 OK ({n_head} heads)")

    os.environ.setdefault("RWKV_V7_ON", "1")
    os.environ.setdefault("RWKV_JIT_ON", "1")
    os.environ.setdefault("RWKV_CUDA_ON", "1" if RWKV_CUDA_KERNEL else "0")

    def _find_nvcc_home():
        for c in sorted(glob.glob("/usr/local/cuda-*"), reverse=True) + ["/usr/local/cuda"]:
            if os.path.exists(os.path.join(c, "bin", "nvcc")):
                return c
        found = shutil.which("nvcc")
        return os.path.dirname(os.path.dirname(found)) if found else None

    if "rwkv.model" not in sys.modules and RWKV_CUDA_KERNEL:
        import torch
        _cuda_home = _find_nvcc_home()
        if _cuda_home is None and torch.version.cuda:
            _major, _minor = torch.version.cuda.split(".")[:2]
            _pkg = f"cuda-nvcc-{_major}-{_minor}"
            print(f"nvcc not found -- installing {_pkg} to match torch.version.cuda={torch.version.cuda!r} ...")
            subprocess.run(["apt-get", "update", "-qq"], check=False)
            subprocess.run(["apt-get", "install", "-y", "-qq", _pkg], check=False)
            _cuda_home = _find_nvcc_home()
        if _cuda_home:
            os.environ["CUDA_HOME"] = _cuda_home
            os.environ["PATH"] = os.path.join(_cuda_home, "bin") + ":" + os.environ["PATH"]
            print(f"CUDA_HOME={_cuda_home} (nvcc found -- fused kernel should build).")
        else:
            print("Could not locate or install nvcc -- falling back to RWKV_CUDA_ON=0 (pure PyTorch).")
            os.environ["RWKV_CUDA_ON"] = "0"

    try:
        from rwkv.model import RWKV
    except Exception as e:
        if os.environ["RWKV_CUDA_ON"] == "1":
            print(f"CUDA kernel build failed even with CUDA_HOME set: {e}\nFalling back to "
                  f"RWKV_CUDA_ON=0 (pure PyTorch) and retrying the import ...")
            os.environ["RWKV_CUDA_ON"] = "0"
            for _mod in [m for m in sys.modules if m == "rwkv" or m.startswith("rwkv.")]:
                del sys.modules[_mod]
            from rwkv.model import RWKV
        else:
            raise
    from rwkv.utils import PIPELINE

    rwkv_model = RWKV(model=str(dest.with_suffix("")), strategy=RWKV_STRATEGY)
    rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")
    _out, _ = rwkv_model.forward(rwkv_pipeline.encode("The capital of France is"), None)
    print(f"  {tag} loaded, strategy={RWKV_STRATEGY!r} -> smoke: "
          f"{rwkv_pipeline.decode([int(_out.argmax())])!r}")


def unload_rwkv():
    global rwkv_model, rwkv_pipeline
    for _name in ("rwkv_model", "rwkv_pipeline"):
        if _name in globals():
            del globals()[_name]
    import torch
    torch.cuda.empty_cache()

print("RWKV checkpoint management ready.")

In [ ]:
import re as _re

def _clean_txt(txt):
    return _re.sub(r"\n{2,}", "\n", txt.replace("\r\n", "\n")).strip()

RWKV_SYSTEM_PREAMBLE = (
    "System: The following is a log of past conversations between you and the user, in "
    "chronological order, each tagged with the date it happened. Remember the details -- names, "
    "dates, preferences, events -- so you can answer questions about them later.")

def render_history_dialogue(haystack_sessions, haystack_dates):
    items = sorted(zip(haystack_dates, haystack_sessions), key=lambda x: x[0])
    lines = [RWKV_SYSTEM_PREAMBLE]
    for date, session_turns in items:
        cleaned = strip_has_answer(session_turns)
        for j, turn in enumerate(cleaned):
            role = "User" if turn.get("role") == "user" else "Assistant"
            content = _clean_txt(str(turn.get("content", "")))
            if not content:
                continue
            if j == 0:
                content = f"[{date}] {content}"
            lines.append(f"{role}: {content}")
    return "\n\n".join(lines)

def render_history_dialogue_truncated(haystack_sessions, haystack_dates, max_tokens):
    items = sorted(zip(haystack_dates, haystack_sessions), key=lambda x: x[0])
    kept, total = [], 0
    for date, turns in reversed(items):
        sess_toks = len(rwkv_pipeline.encode(render_history_dialogue([turns], [date])))
        if total + sess_toks > max_tokens and kept:
            break
        kept.append((date, turns)); total += sess_toks
    kept.reverse()
    return render_history_dialogue([t for _, t in kept], [d for d, _ in kept])


import torch

@torch.no_grad()
def feed(text, state=None):
    toks = rwkv_pipeline.encode(text)
    if not toks:
        return None, state, 0
    out = None
    for i in range(0, len(toks), CHUNK_TOKENS):
        out, state = rwkv_model.forward(toks[i:i + CHUNK_TOKENS], state)
    return out, state, len(toks)


@torch.no_grad()
def probe_memory(state, instruction, max_new=NOTE_MAX_TOKENS):
    s = [t.clone() for t in state] if state is not None else None
    probe_text = f"\n\nUser: {instruction}\n\nAssistant:"
    out, s = rwkv_model.forward(rwkv_pipeline.encode(probe_text), s)
    toks, recent = [], []
    for _ in range(max_new):
        t = int(out.argmax())
        if t == 0:
            break
        toks.append(t); recent.append(t)
        if len(recent) > 16:
            recent.pop(0)
        if len(recent) == 16 and recent[:8] == recent[8:]:
            toks = toks[:-8]
            break
        out, s = rwkv_model.forward([t], s)
    text = rwkv_pipeline.decode(toks)
    for stop in ("\n\nUser:", "\nUser:"):
        idx = text.find(stop)
        if idx != -1:
            text = text[:idx]
    return text.strip()


def probe_memorag(state, question, question_date, kind):
    instruction = f"Today's date is {question_date}. " + MEMORAG_PROMPTS[kind].format(question=question)
    return probe_memory(state, instruction)

print("feed() / probe_memory() / probe_memorag() / render_history_dialogue*() defined.")

In [ ]:
_gate_a_done = False

def gate_a_correctness():
    print("Gate A: correctness checks before the real run.\n")
    _, _smoke_state, _n = feed(
        "System: You are a helpful assistant with a good memory.\n\nUser: My favorite color is teal.\n\n"
        "Assistant: Got it, I'll remember that your favorite color is teal.", None)
    _smoke_note = probe_memory(_smoke_state, "What is my favorite color?", max_new=32)
    print(f"(a) smoke test ({_n} tokens fed): {_smoke_note!r}")
    assert "teal" in _smoke_note.lower(), (
        f"RWKV did not recall a fact from a 2-sentence conversation. Got: {_smoke_note!r}")
    print("    -> PASSED (recalled 'teal').\n")

    _probe_text = ("The quick brown fox jumps over the lazy dog. " * 400).strip()
    _probe_toks = rwkv_pipeline.encode(_probe_text)
    print(f"(b) chunking equivalence probe: {len(_probe_toks):,} tokens, CHUNK_TOKENS={CHUNK_TOKENS}")
    _out_whole, _ = rwkv_model.forward(_probe_toks, None)
    _out_chunked, _, _n_chunked = feed(_probe_text, None)
    _argmax_whole, _argmax_chunked = int(_out_whole.argmax()), int(_out_chunked.argmax())
    assert _n_chunked == len(_probe_toks)
    assert _argmax_whole == _argmax_chunked, (
        f"Chunked feed() disagrees with an unchunked forward() on next-token argmax "
        f"({_argmax_chunked} vs {_argmax_whole}).")
    print(f"    -> PASSED (both argmax to {_argmax_whole} = {rwkv_pipeline.decode([_argmax_whole])!r}).\n")
    print("Gate A PASSED.")


def gate_b_throughput(tag):
    # Same measurement longmemeval_memorag.ipynb's own Gate B makes (one real haystack), per tag,
    # hard-asserted against a budget rather than only printed, since this may run unattended.
    bench_q = QID2S[QUESTION_IDS[0]]
    hist = render_history_dialogue(bench_q["haystack_sessions"], bench_q["haystack_dates"])
    t0 = time.time(); _, bs, n_fed = feed(hist, None); build_s = time.time() - t0
    t0 = time.time()
    probe_memorag(bs, bench_q["question"], bench_q["question_date"], "sur")
    probe_memorag(bs, bench_q["question"], bench_q["question_date"], "span")
    probe_memorag(bs, bench_q["question"], bench_q["question_date"], "qa")
    probe_s = time.time() - t0
    del bs
    per_q_s = build_s + probe_s
    projected_h = len(QUESTION_IDS) * per_q_s / 3600.0
    tok_per_s = n_fed / build_s if build_s > 0 else float("nan")
    print(f"  {tag}: {n_fed:,} tok fed in {build_s:.1f}s ({tok_per_s:,.0f} tok/s); 3 probes in "
          f"{probe_s:.1f}s -> extrapolated {projected_h:.2f}h for {len(QUESTION_IDS)} questions")
    budget = STAGE_A_BUDGET_H[tag]
    assert projected_h <= budget, (
        f"{tag}: extrapolated {projected_h:.2f}h exceeds its {budget}h budget -- STOPPING before "
        f"the full memory-build loop. Either raise STAGE_A_BUDGET_H[{tag!r}] deliberately, drop "
        f"this tag from RWKV_ACTIVE, or check why throughput is low (RWKV_CUDA_ON actually took "
        f"effect? see the loaded-strategy print above).")
    return projected_h


def memory_path(tag, source="s"):
    # g1i-2.9b reuses longmemeval_memorag.ipynb's OWN cached files, read-only, if present -- never
    # writes through this path for that tag. Falls back to this notebook's own ext/store if that
    # notebook never ran (so this works standalone either way).
    if tag == "g1i-2.9b":
        shared = STORE / f"memory_{source}.jsonl"
        if shared.exists():
            return shared
    return MEM_STORE / f"memory_{source}__{tag}.jsonl"


def build_rwkv_memory(tag, source, question_ids, render_fn):
    global _gate_a_done
    out_path = memory_path(tag, source)
    done_ids = set()
    if out_path.exists():
        for line in out_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                done_ids.add(json.loads(line)["question_id"])
    todo = [q for q in question_ids if q not in done_ids]
    if not todo:
        print(f"  {tag}/{source}: all {len(question_ids)} memory rows already built, skipping.")
        return
    load_rwkv(tag)
    if not _gate_a_done:
        gate_a_correctness()
        _gate_a_done = True
    gate_b_throughput(tag)
    print(f"  {tag}/{source}: building {len(todo)} of {len(question_ids)} (have {len(done_ids)}) ...")
    from tqdm.auto import tqdm
    with open(out_path, "a", encoding="utf-8") as out_f:
        for qid in tqdm(todo, desc=f"rwkv memory ({tag}/{source})"):
            q = QID2S[qid]
            history = render_fn(q["haystack_sessions"], q["haystack_dates"])
            t0 = time.time(); _, state, n_fed = feed(history, None); build_s = time.time() - t0
            t0 = time.time()
            sur = probe_memorag(state, q["question"], q["question_date"], "sur")
            span = probe_memorag(state, q["question"], q["question_date"], "span")
            answer = probe_memorag(state, q["question"], q["question_date"], "qa")
            probe_s = time.time() - t0
            del state
            out_f.write(json.dumps({"question_id": qid, "sur": sur, "span": span, "answer": answer,
                                     "tokens_fed": n_fed, "build_s": build_s, "probe_s": probe_s}) + "\n")
            out_f.flush()
    unload_rwkv()


def load_memory_tag(tag, source="s"):
    p = memory_path(tag, source)
    out = {}
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line); out[r["question_id"]] = r
    return out

print("Stage A functions ready.")

In [ ]:
MEMORY = {}
if CAN_RUN_RWKV:
    for _tag in RWKV_ACTIVE:
        build_rwkv_memory(_tag, "s", QUESTION_IDS, render_history_dialogue)
    build_rwkv_memory("g1i-2.9b", "s_trunc16k", QUESTION_IDS,
                      lambda sess, dates: render_history_dialogue_truncated(sess, dates, RWKV_TRUNCATE_TOKENS))
    del _tag

    for _tag in RWKV_ACTIVE:
        MEMORY[_tag] = load_memory_tag(_tag, "s")
        _missing = [q for q in QUESTION_IDS if q not in MEMORY[_tag]]
        print(f"  {_tag}: {len(MEMORY[_tag])} rows, {len(_missing)} missing "
              f"({'OK' if not _missing else 'INCOMPLETE -- re-run the cell above'})")
    MEMORY["g1i-2.9b_trunc16k"] = load_memory_tag("g1i-2.9b", "s_trunc16k")
    print(f"  g1i-2.9b_trunc16k: {len(MEMORY['g1i-2.9b_trunc16k'])} rows")
    del _tag, _missing
else:
    print("SKIP: RWKV unavailable on this GPU -- RNN-dependent arms will be skipped downstream.")


def gate_clue_sanity():
    print("Gate E4: each RWKV tag produced its own, non-degenerate clues.\n")
    ok = True
    tags = [t for t in RWKV_ACTIVE if MEMORY.get(t)]
    for tag in tags:
        mem = MEMORY[tag]
        nonempty = sum(1 for q in QUESTION_IDS
                       if mem.get(q) and mem[q].get("sur", "").strip() and mem[q].get("span", "").strip())
        frac = nonempty / len(QUESTION_IDS)
        ok &= frac >= 0.95
        print(f"  [{'OK' if frac >= 0.95 else 'FAIL'}] {tag}: {frac:.0%} of questions have "
              f"non-empty sur+span clues")
    if len(tags) >= 2:
        sample = QUESTION_IDS[:5]
        distinct = sum(1 for qid in sample
                       if len({MEMORY[t][qid]["span"] for t in tags if qid in MEMORY[t]}) > 1)
        ok &= distinct >= 1
        print(f"  [{'OK' if distinct >= 1 else 'FAIL'}] {distinct}/{len(sample)} sample questions "
              f"have DIFFERENT 'span' clues across RWKV sizes")
    print(f"\nGate E4: {'PASSED' if ok else 'FAILED'}")
    return ok

gate_clue_sanity()

## Section 11: Stage B -- the retrieval layer

Transcribed from `longmemeval_memorag.ipynb`'s own Section 11 verbatim, including the three Stella
compat patches -- this notebook loads Stella in its own kernel, so these are needed here too
regardless of what the base notebook already fixed (a genuine, unavoidable duplication given a
notebook can't import another; if the base notebook's fix ever changes, this copy needs the same
change). Points at the SAME `STORE`, so the sha1-keyed embedding cache (`emb_stella.*`) is shared
and extended, not duplicated.

In [ ]:
class EmbeddingCache:
    def __init__(self, name, dim=None):
        self.name = name
        self.idx_path, self.mat_path = STORE / f"emb_{name}.json", STORE / f"emb_{name}.npy"
        self.index = json.loads(self.idx_path.read_text()) if self.idx_path.exists() else {}
        self.mat = np.load(self.mat_path) if self.mat_path.exists() else (
            np.zeros((0, dim), dtype=np.float32) if dim else None)
        self._dirty = False

    def get_or_encode(self, texts, encode_fn, batch=64):
        missing = []
        for t in texts:
            h = hashlib.sha1(t.encode("utf-8")).hexdigest()
            if h not in self.index and h not in {m[0] for m in missing}:
                missing.append((h, t))
        if missing:
            vecs = encode_fn([t for _, t in missing], batch)
            if self.mat is None or self.mat.shape[0] == 0:
                self.mat = np.asarray(vecs, dtype=np.float32)
                start = 0
            else:
                start = self.mat.shape[0]
                self.mat = np.vstack([self.mat, np.asarray(vecs, dtype=np.float32)])
            for j, (h, _) in enumerate(missing):
                self.index[h] = start + j
            self._dirty = True
        rows = [self.index[hashlib.sha1(t.encode("utf-8")).hexdigest()] for t in texts]
        return self.mat[rows]

    def save(self):
        if not self._dirty:
            return
        self.idx_path.write_text(json.dumps(self.index))
        np.save(self.mat_path, self.mat)
        self._dirty = False


def _patch_qwen2_rope_theta_compat():
    from transformers.models.qwen2.configuration_qwen2 import Qwen2Config
    try:
        Qwen2Config(rope_theta=1.0).rope_theta
    except AttributeError:
        def _get(self):
            return (getattr(self, "rope_parameters", None) or {}).get("rope_theta", 10000.0)
        def _set(self, value):
            rp = getattr(self, "rope_parameters", None) or {}
            rp["rope_theta"] = value
            self.rope_parameters = rp
        Qwen2Config.rope_theta = property(_get, _set)


def _patch_dynamic_cache_legacy_compat():
    from transformers import DynamicCache
    if not hasattr(DynamicCache, "from_legacy_cache"):
        @classmethod
        def _from_legacy_cache(cls, past_key_values=None):
            cache = cls()
            if past_key_values is not None:
                for layer_idx in range(len(past_key_values)):
                    key_states, value_states = past_key_values[layer_idx]
                    cache.update(key_states, value_states, layer_idx)
            return cache
        DynamicCache.from_legacy_cache = _from_legacy_cache
    if not hasattr(DynamicCache, "get_usable_length"):
        def _get_usable_length(self, new_seq_length, layer_idx=0):
            return self.get_seq_length(layer_idx)
        DynamicCache.get_usable_length = _get_usable_length
    if not hasattr(DynamicCache, "to_legacy_cache"):
        def _to_legacy_cache(self):
            return tuple((layer.keys, layer.values) for layer in self.layers)
        DynamicCache.to_legacy_cache = _to_legacy_cache


def _fix_rotary_embedding_buffers(hf_model):
    # Stella's frozen (2024) modeling_qwen.py predates transformers' `original_inv_freq`-based
    # meta-device repair mechanism -- its Qwen2RotaryEmbedding buffers come out of from_pretrained
    # as garbage on every load. Recompute them directly from the module's own base/dim.
    device = next(hf_model.parameters()).device
    dtype = next(hf_model.parameters()).dtype
    fixed = 0
    for module in hf_model.modules():
        if type(module).__name__ == "Qwen2RotaryEmbedding" and hasattr(module, "inv_freq"):
            module.inv_freq = 1.0 / (module.base ** (
                torch.arange(0, module.dim, 2, dtype=torch.int64).float().to(device) / module.dim))
            module._set_cos_sin_cache(seq_len=module.max_position_embeddings, device=device, dtype=dtype)
            fixed += 1
    assert fixed > 0, "found zero Qwen2RotaryEmbedding submodules to fix -- re-check modeling_qwen.py live."
    return fixed


class DenseRetriever:
    def __init__(self, kind):
        self.kind = kind
        self.cache = EmbeddingCache(kind)
        self._model = None

    def _load(self):
        if self._model is not None:
            return
        _patch_qwen2_rope_theta_compat()
        _patch_dynamic_cache_legacy_compat()
        from sentence_transformers import SentenceTransformer
        import warnings
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*", category=FutureWarning)
            self._model = SentenceTransformer(
                STELLA_HF, trust_remote_code=True, device="cuda",
                model_kwargs={"torch_dtype": torch.float32})
            _fix_rotary_embedding_buffers(self._model[0].auto_model)
        _PROBE_TEXTS = ["nan probe short", "nan probe a considerably longer sentence than the first one, to force padding"]
        probe = self._model.encode(_PROBE_TEXTS, show_progress_bar=False)
        assert np.isfinite(probe).all(), (
            f"{self.kind}: still produced non-finite embeddings after fixing the RoPE buffers.")
        print(f"  [ok] {self.kind}: embeddings are finite after fixing RoPE buffers.")

    def _encode(self, texts, batch, is_query=False):
        self._load()
        kw = {"prompt_name": "s2p_query"} if is_query else {}
        return self._model.encode(texts, batch_size=batch, show_progress_bar=False,
                                  normalize_embeddings=True, **kw)

    def rank(self, query, key_texts):
        docs = self.cache.get_or_encode(key_texts, lambda t, b: self._encode(t, b, False))
        qv = self._encode([query], 1, True)[0]
        scores = docs @ qv
        return list(np.argsort(-scores))


class BM25Retriever:
    kind = "bm25"
    @staticmethod
    def _tok(s):
        return _re.findall(r"[a-z0-9]+", s.lower())

    def rank(self, query, key_texts):
        from rank_bm25 import BM25Okapi
        bm = BM25Okapi([self._tok(t) for t in key_texts])
        return list(np.argsort(-bm.get_scores(self._tok(query))))


_RETRIEVERS = {}
def get_retriever(kind):
    if kind not in _RETRIEVERS:
        _RETRIEVERS[kind] = BM25Retriever() if kind == "bm25" else DenseRetriever(kind)
    return _RETRIEVERS[kind]

def save_all_retriever_caches():
    for retr in _RETRIEVERS.values():
        cache = getattr(retr, "cache", None)
        if cache is not None:
            cache.save()

print("retrieval stack ready:", ", ".join(RETRIEVERS))

In [ ]:
def gate3_retrieval_harness(sample_n=15):
    ok = True
    print("Gate 3: retrieval harness self-check.\n")
    if CAN_RUN_EMBEDDINGS:
        probe_texts = ["hello world", "a second short probe sentence", "hello world"]
        cache = EmbeddingCache("gate3_probe")
        retr = get_retriever(PRIMARY_DENSE_RETRIEVER)
        v1 = cache.get_or_encode(probe_texts, lambda t, b: retr._encode(t, b, False))
        v2 = cache.get_or_encode(probe_texts, lambda t, b: retr._encode(t, b, False))
        n_bad = int((~np.isfinite(v1)).sum())
        if n_bad:
            ok = False
            print(f"  [FAIL] {PRIMARY_DENSE_RETRIEVER} produced {n_bad}/{v1.size} non-finite values.")
        else:
            roundtrip_ok = np.allclose(v1, v2) and np.allclose(v1[0], v1[2])
            ok &= roundtrip_ok
            print(f"  [{'OK' if roundtrip_ok else 'FAIL'}] embedding cache round-trips and dedupes "
                  f"identical text to the identical vector")
    else:
        print("  [SKIP] embeddings unavailable on this GPU -- cache round-trip not checked")

    sample = QUESTION_IDS[:sample_n]
    hits = 0
    for qid in sample:
        q = QID2S[qid]
        ids, values, keys, stamps = build_corpus(q, "session")
        correct = evidence_ids(q, "session")
        if not correct:
            continue
        ranking = get_retriever("bm25").rank(q["question"], keys)
        _, recall_all, _ = evaluate_retrieval(ranking, correct, ids, k=10)
        hits += 1 if recall_all or any(ids[i] in correct for i in ranking[:10]) else 0
    print(f"  [info] plain-question BM25 top-10 contained an oracle session for {hits}/{len(sample)} "
          f"sample questions.")
    print(f"\nGate 3: {'PASSED' if ok else 'CHECK WARNINGS ABOVE'}")
    return ok

gate3_retrieval_harness()

## Section 12: Query construction and chunk selection

`build_query_set`/`fuse`/`select_chunks` transcribed from `longmemeval_memorag.ipynb`'s own
Section 11 -- `memory_source` is simply a `MEMORY` dict key now (`"g1i-1.5b"`, `"g1i-2.9b"`,
`"g1i-2.9b_trunc16k"`, ...), directly, since this notebook has no base-registry bare `"s"` key to
alias around.

In [ ]:
def build_query_set(qid, design, memory_source):
    q = QID2S[qid]["question"]
    mem = MEMORY.get(memory_source, {}).get(qid)
    out = []
    if design:
        for comp in design.split("+"):
            if comp == "ans":
                if mem and mem.get("answer", "").strip():
                    out.append(mem["answer"].strip())
                continue
            text = (mem or {}).get(comp, "")
            out += [s.strip() for s in text.split("\n") if len(s.split()) > 3]
    out.append(q)
    return out


def fuse(rankings_per_query, n_docs, method, topk):
    if method == "union_literal":
        pool = set()
        for r in rankings_per_query:
            pool |= set(r[:topk])
        return sorted(pool)
    if method == "rank":
        best_pos = {}
        for r in rankings_per_query:
            for pos, idx in enumerate(r):
                if idx not in best_pos or pos < best_pos[idx]:
                    best_pos[idx] = pos
        order = sorted(range(n_docs), key=lambda i: best_pos.get(i, n_docs))
        return order[:topk]
    if method == "rrf":
        RRF_K = 60
        scores = [0.0] * n_docs
        for r in rankings_per_query:
            for pos, idx in enumerate(r):
                scores[idx] += 1.0 / (RRF_K + pos + 1)
        order = sorted(range(n_docs), key=lambda i: -scores[i])
        return order[:topk]
    raise NotImplementedError(method)


def select_chunks(q, arm):
    ids, values, keys, stamps = build_corpus(q, arm.granularity)
    if arm.retriever == "none":
        chunks = list(zip(stamps, values))
        return (chunks if not arm.topk else chunks[-arm.topk:]), ids
    queries = build_query_set(q["question_id"], arm.query_design, arm.memory_source)
    retr = get_retriever(arm.retriever)
    rankings = [retr.rank(qq, keys) for qq in queries]
    picked_idx = fuse(rankings, len(ids), arm.fusion, arm.topk)
    picked, total = [], 0
    for i in picked_idx:
        if arm.token_budget:
            t = len(_re.findall(r"\S+", json.dumps(values[i])))
            if total + t > arm.token_budget and picked:
                break
            total += t
        picked.append((stamps[i], values[i]))
    if "ans" in arm.query_design.split("+"):
        mem = MEMORY.get(arm.memory_source, {}).get(q["question_id"])
        if mem and mem.get("answer", "").strip():
            picked.append((q["question_date"], f"The answer might be {mem['answer'].strip()}."))
    return picked, ids


def memonly_chunks(q, memory_source):
    mem = MEMORY.get(memory_source, {}).get(q["question_id"])
    if mem:
        text = f"Facts recalled: {mem['span']}\nClue questions: {mem['sur']}\nDraft answer: {mem['answer']}"
    else:
        text = "(no memory available)"
    return [(q["question_date"], text)]


def prompt_for(qid, arm, reader):
    if arm.source == "closedbook":
        q = QID2S[qid]
        return PROMPT_TEMPLATE_CLOSEDBOOK.format(q["question_date"], q["question"])
    q = (QID2ORACLE if arm.source == "oracle" else QID2S)[qid]
    if arm.source == "memonly":
        chunks = memonly_chunks(q, arm.memory_source)
        return build_prompt(q, chunks, reader, arm.reading, arm.history_format,
                            merge_mode="replace", granularity="session")
    chunks, _ = select_chunks(q, arm)
    return build_prompt(q, chunks, reader, arm.reading, arm.history_format, "none", arm.granularity)

print("Query construction ready.")

## Section 13: The arm registry

15 arms: 5 RNN-independent (shared across every RWKV size) + 3 per RWKV tag x 3 active tags + 1
truncated-context ablation (2.9B only). No `ext_` prefix needed -- this notebook owns its whole
registry, but the RWKV tag is still baked into every memory-dependent arm name so it's always
obvious from a filename which model produced it.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Arm:
    name: str
    source: str = "s"                 # "oracle" | "s" | "closedbook" | "memonly"
    query_design: str = ""            # "" | "+"-joined subset of {"sur","span","ans"}
    memory_source: str = "g1i-2.9b"   # a MEMORY dict key
    retriever: str = "none"           # "none" | "stella" | "bm25"
    granularity: str = "session"
    fusion: str = "rank"
    topk: int = 10
    token_budget: int = 0
    reading: str = "direct"
    history_format: str = "json"
    artifact: str = ""


def build_arm_registry():
    arms = []
    arms.append(Arm("closedbook", source="closedbook", artifact="core"))
    arms.append(Arm("oracle_con", source="oracle", reading="con", topk=0, artifact="core"))
    arms.append(Arm("s_con", source="s", reading="con", topk=0, artifact="core"))
    for k in (5, 10):
        arms.append(Arm(f"rag_q_top{k}", retriever=PRIMARY_DENSE_RETRIEVER, query_design="",
                        topk=k, artifact="core"))
    for tag in RWKV_ACTIVE:
        arms.append(Arm(f"memonly__{tag}", source="memonly", memory_source=tag, artifact="rnn"))
        for k in (5, 10):
            arms.append(Arm(f"memorag_top{k}__{tag}", retriever=PRIMARY_DENSE_RETRIEVER,
                            query_design=QUERY_DESIGN, memory_source=tag, topk=k, artifact="rnn"))
    arms.append(Arm("memorag_trunc_top10__g1i-2.9b", retriever=PRIMARY_DENSE_RETRIEVER,
                    query_design=QUERY_DESIGN, memory_source="g1i-2.9b_trunc16k", topk=10,
                    artifact="rnn"))
    return {a.name: a for a in arms}

ARMS = build_arm_registry()
by_artifact = Counter(a.artifact for a in ARMS.values())
print(f"{len(ARMS)} arms x {len(READER_ORDER)} readers x {len(QUESTION_IDS)} questions = "
      f"{len(ARMS) * len(READER_ORDER) * len(QUESTION_IDS):,} generations")
for k, v in by_artifact.items():
    print(f"   {k:10s} {v:3d}")

assert len(ARMS) == len({a.name for a in ARMS.values()}), "duplicate arm name in the registry"
print("\nGate: arm names are unique -- OK.")

## Section 14: Gate 4 -- every arm builds a valid prompt, for every reader

In [ ]:
def gate4_prompt_smoke(n=2):
    print(f"Gate 4: prompt-construction smoke test -- {len(ARMS)} arms x {len(READER_ORDER)} "
          f"readers x {n} questions, no model call.\n")
    bad = []
    for reader in READER_ORDER:
        for qid in QUESTION_IDS[:n]:
            for arm in ARMS.values():
                try:
                    prompt = prompt_for(qid, arm, reader)
                except Exception as e:
                    bad.append((reader, arm.name, qid, f"raised {type(e).__name__}: {e}"))
                    continue
                if not prompt or not prompt.strip():
                    bad.append((reader, arm.name, qid, "empty prompt"))
                    continue
                if "has_answer" in prompt:
                    bad.append((reader, arm.name, qid, "LEAKED 'has_answer' into the rendered prompt"))
                    continue
                n_tok = count_tokens(prompt, reader)
                budget = max_retrieval_length(reader, arm.reading) + GEN_LENGTH_BY_READING[arm.reading] + 1000
                if n_tok > budget + 50:
                    bad.append((reader, arm.name, qid, f"{n_tok} tokens exceeds budget ~{budget}"))
    if bad:
        print(f"  {len(bad)} FAILURES:")
        for reader, name, qid, reason in bad[:20]:
            print(f"      {reader:24s} {name:28s} {qid:10s} {reason}")
        raise AssertionError(f"Gate 4 found {len(bad)} problem(s) -- fix before Stage C.")
    print(f"  all {len(ARMS)} arms x {len(READER_ORDER)} readers x {n} questions produced a valid, "
          f"budget-respecting, leak-free prompt.")
    save_all_retriever_caches()
    print("\nGate 4 PASSED.")

gate4_prompt_smoke()

## Section 15: Generation backends

`OpenRouterBackend` transcribed verbatim from `longmemeval_recreation.ipynb`, `LocalVLLMBackend`
transcribed from `longmemeval_memorag.ipynb`'s own Section 14 (including its free-memory-based
`gpu_memory_utilization` sizing, needed because Stella stays resident through Stage C alongside
the local 1B engine).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass as _dc

@_dc
class GenRecord:
    question_id: str
    hypothesis: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    cost_usd: float = 0.0
    latency_s: float = 0.0
    provider: str = ""
    error: str = ""


class OpenRouterBackend:
    def __init__(self, reader):
        self.reader, self.spec = reader, READERS[reader]

    def _shrink(self, prompt, factor):
        marker, tail_marker = "History Chats:\n\n", "\n\nCurrent Date: "
        i, j = prompt.find(marker), prompt.rfind(tail_marker)
        if i < 0 or j < 0 or j <= i:
            return prompt[:int(len(prompt) * factor)]
        head, hist, tail = prompt[:i + len(marker)], prompt[i + len(marker):j], prompt[j:]
        return head + hist[:max(1, int(len(hist) * factor))] + tail

    def _one(self, qid, prompt, max_tokens, retries=5):
        body = {"provider": {"order": [self.spec["provider"]], "allow_fallbacks": False}}
        factor, sent = 1.0, prompt
        for attempt in range(retries):
            t0 = time.time()
            try:
                r = openrouter_client.chat.completions.create(
                    model=self.spec["or_id"],
                    messages=[{"role": "user", "content": sent}],
                    temperature=TEMPERATURE, max_tokens=max_tokens, n=1,
                    extra_body=body,
                )
                usage = r.usage
                return GenRecord(
                    question_id=qid, hypothesis=(r.choices[0].message.content or "").strip(),
                    prompt_tokens=getattr(usage, "prompt_tokens", 0),
                    completion_tokens=getattr(usage, "completion_tokens", 0),
                    cost_usd=float(getattr(usage, "cost", 0.0) or 0.0),
                    latency_s=time.time() - t0,
                    provider=getattr(r, "provider", self.spec["provider"]))
            except Exception as e:
                msg = str(e).lower()
                too_long = any(s in msg for s in ("context length", "context_length", "maximum context",
                                                   "too long", "max_tokens", "token limit", "input length"))
                if too_long and factor > 0.2:
                    factor *= 0.75
                    sent = self._shrink(prompt, factor)
                    continue
                if attempt == retries - 1:
                    return GenRecord(question_id=qid, hypothesis="", latency_s=time.time() - t0,
                                     provider=self.spec["provider"], error=repr(e)[:300])
                time.sleep(2 ** attempt)

    def generate(self, items, max_tokens):
        with ThreadPoolExecutor(max_workers=OR_CONCURRENCY) as pool:
            return list(pool.map(lambda it: self._one(it[0], it[1], max_tokens), items))


class LocalVLLMBackend:
    def __init__(self, reader):
        self.reader, self.spec = reader, READERS[reader]
        os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
        from vllm import LLM
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        headroom_gb = 6.0
        target_gb = max((free_bytes / 1e9) - headroom_gb, (total_bytes / 1e9) * 0.3)
        gpu_mem_util = min(0.85, target_gb / (total_bytes / 1e9))
        print(f"  GPU memory: {free_bytes/1e9:.1f}GB free / {total_bytes/1e9:.1f}GB total -- "
              f"requesting gpu_memory_utilization={gpu_mem_util:.2f} (reserving ~{headroom_gb:.0f}GB "
              f"headroom for Stella, which stays loaded)")
        print(f"  loading {self.spec['hf_id']} into vLLM ...")
        self.llm = LLM(model=self.spec["hf_id"], max_model_len=self.spec["served_context"],
                       gpu_memory_utilization=gpu_mem_util, dtype="bfloat16", enforce_eager=False)

    def generate(self, items, max_tokens):
        from vllm import SamplingParams
        tok = self.llm.get_tokenizer()
        prompts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                           tokenize=False, add_generation_prompt=True)
                   for _, p in items]
        sp = SamplingParams(temperature=TEMPERATURE, max_tokens=max_tokens, n=1)
        t0 = time.time()
        outs = self.llm.generate(prompts, sp)
        dt = (time.time() - t0) / max(1, len(items))
        return [GenRecord(question_id=qid, hypothesis=o.outputs[0].text.strip(),
                          prompt_tokens=len(o.prompt_token_ids),
                          completion_tokens=len(o.outputs[0].token_ids), latency_s=dt,
                          provider="local-vllm")
                for (qid, _), o in zip(items, outs)]

    def close(self):
        try:
            self.llm.llm_engine.engine_core.shutdown()
        except Exception:
            pass
        if hasattr(self, "llm"):
            del self.llm
        import torch._dynamo
        torch._dynamo.reset()
        from vllm.distributed.parallel_state import cleanup_dist_env_and_memory
        cleanup_dist_env_and_memory()


_or_backends = {}

def backend_for(reader):
    spec = READERS[reader]
    if spec["route"] == "openrouter":
        if reader not in _or_backends:
            _or_backends[reader] = OpenRouterBackend(reader)
        return _or_backends[reader]
    return LocalVLLMBackend(reader)


def _load_arm_records(out_path):
    recs = {}
    if out_path.exists():
        for line in out_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line)
                prev = recs.get(r["question_id"])
                if prev is None or (not prev.get("hypothesis") and r.get("hypothesis")):
                    recs[r["question_id"]] = r
    return recs


def run_arm(backend, reader, arm, question_ids, chunk_size=16):
    out_path = RUNS / reader / f"{arm.name}.jsonl"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    recs = _load_arm_records(out_path)
    done = {q for q, r in recs.items() if r.get("hypothesis")}
    todo = [i for i in question_ids if i not in done]
    if not todo:
        return 0
    max_tokens = GEN_LENGTH_BY_READING[arm.reading]
    written = 0
    for start in range(0, len(todo), chunk_size):
        batch = todo[start:start + chunk_size]
        items = [(qid, prompt_for(qid, arm, reader)) for qid in batch]
        for rec in backend.generate(items, max_tokens):
            recs[rec.question_id] = rec.__dict__ | {"reader": reader, "arm": arm.name, "artifact": arm.artifact}
            written += 1
        tmp = out_path.with_suffix(".jsonl.tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            for qid in question_ids:
                if qid in recs:
                    f.write(json.dumps(recs[qid]) + "\n")
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, out_path)
        save_all_retriever_caches()
    return written

print("Generation backends ready:", ", ".join(f"{r} [{READERS[r]['route']}]" for r in READER_ORDER))

## Section 16: Gate E5 -- OpenRouter smoke test, then Stage C -- running the sweep

In [ ]:
def gate_e5_openrouter_smoke():
    print("Gate E5: OpenRouter smoke test (llama-3.1-8b-instruct via DeepInfra)\n")
    backend = OpenRouterBackend("llama-3.1-8b-instruct")
    rec = backend._one("smoke_short", "Reply with exactly the single word: OK", max_tokens=5)
    print(f"  [{'OK' if not rec.error else 'FAIL'}] short call: provider={rec.provider!r} "
          f"hypothesis={rec.hypothesis!r} error={rec.error!r}")
    q = QID2S[QUESTION_IDS[0]]
    ids, values, keys, stamps = build_corpus(q, "session")
    prompt = build_prompt(q, list(zip(stamps, values)), "llama-3.1-8b-instruct", "con")
    n_tok = count_tokens(prompt, "llama-3.1-8b-instruct")
    rec2 = backend._one("smoke_long", prompt, max_tokens=50)
    ok_long = not rec2.error
    print(f"  [{'OK' if ok_long else 'FAIL'}] ~{n_tok:,}-token S-arm prompt: "
          f"{'accepted' if ok_long else rec2.error}")
    ok = (not rec.error) and ok_long
    print(f"\nGate E5: {'PASSED' if ok else 'FAILED'}")
    return ok

gate_e5_openrouter_smoke()

In [ ]:
def arm_needs_memory(arm):
    return arm.source == "memonly" or (arm.query_design and any(
        c in arm.query_design.split("+") for c in ("sur", "span", "ans")))

def arm_available(arm):
    if not arm_needs_memory(arm):
        return True
    return bool(MEMORY.get(arm.memory_source))


def run_all():
    runnable = {n: a for n, a in ARMS.items() if arm_available(a)}
    skipped = {n: a for n, a in ARMS.items() if n not in runnable}
    if skipped:
        print(f"SKIPPING {len(skipped)} memory-dependent arms (RWKV memory unavailable): "
              f"{', '.join(skipped)}")

    for reader in READER_ORDER:
        spec = READERS[reader]
        if spec["route"] == "local" and not CAN_RUN_LOCAL_READER:
            print(f"SKIP {reader}: local reader unavailable on this GPU.")
            continue
        print(f"\n=== {reader} [{spec['route']}] ===")
        backend = backend_for(reader)

        arm0 = next(iter(runnable.values()))
        run_arm(backend, reader, arm0, QUESTION_IDS[:2])
        recs0 = _load_arm_records(RUNS / reader / f"{arm0.name}.jsonl")
        assert all(recs0[q]["hypothesis"] for q in QUESTION_IDS[:2] if q in recs0), (
            f"{reader}/{arm0.name}: smoke generation produced an empty hypothesis -- stopping.")

        total = len(runnable)
        for i, (name, arm) in enumerate(runnable.items(), 1):
            n = run_arm(backend, reader, arm, QUESTION_IDS)
            if n:
                print(f"  [{i:3d}/{total}] {name:32s} +{n} generations")
        if spec["route"] == "local":
            backend.close()
    print("\nGeneration sweep complete.")

run_all()

## Section 17: Stage D -- judging with GPT-4o

In [ ]:
JUDGE_TPM           = 30000     # placeholder -- probe_judge_tpm() below measures the real value
JUDGE_TPM_HEADROOM  = 0.85
JUDGE_CONCURRENCY   = 8
JUDGE_MAX_ATTEMPTS  = 8
PRICE_IN_PER_MTOK, PRICE_OUT_PER_MTOK = 2.50, 10.00     # gpt-4o-2024-08-06, USD/1M tokens
CONFIRM_JUDGE_SPEND = True      # set False for a priced dry-run that sends nothing
JUDGE_LIMIT_QUESTIONS = None

import threading
from collections import deque

def iter_generations():
    for reader_dir in sorted(RUNS.iterdir()):
        if not reader_dir.is_dir():
            continue
        for f in sorted(reader_dir.glob("*.jsonl")):
            for line in f.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    yield json.loads(line)

def judge_key(rec):
    return f"{rec['reader']}|{rec['arm']}|{rec['question_id']}"

def build_judge_request(rec):
    q = QID2S[rec["question_id"]]
    abstention = rec["question_id"].endswith("_abs")
    prompt = judge_prompt(q["question_type"], q["question"], q["answer"], rec["hypothesis"], abstention)
    return {"model": JUDGE_MODEL, "messages": [{"role": "user", "content": prompt}],
            "temperature": 0, "max_tokens": 10, "n": 1}

def judge_tokens(rec):
    return len(build_judge_request(rec)["messages"][0]["content"]) // 4 + 10

JUDGED_PATH = RESULTS / "judged.jsonl"

def load_judged():
    out = {}
    if JUDGED_PATH.exists():
        for line in JUDGED_PATH.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line); out[r["key"]] = r["label"]
    return out

def _scope_qids():
    return set(QUESTION_IDS if JUDGE_LIMIT_QUESTIONS is None else QUESTION_IDS[:JUDGE_LIMIT_QUESTIONS])

def estimate_judge_cost(pending, sample=400):
    if not pending:
        return 0.0, 0.0
    step = max(1, len(pending) // sample)
    sampled = pending[::step][:sample]
    chars = sum(len(build_judge_request(r)["messages"][0]["content"]) for r in sampled)
    in_tok = (chars / len(sampled)) / 4.0 * len(pending)
    out_tok = 10 * len(pending)
    usd = in_tok / 1e6 * PRICE_IN_PER_MTOK + out_tok / 1e6 * PRICE_OUT_PER_MTOK
    return in_tok, usd

print("Judging scaffolding ready.")

In [ ]:
class TokenPacer:
    def __init__(self, tpm, headroom=0.85):
        self.budget = max(1000, int(tpm * headroom))
        self.events = deque()
        self.spent = 0
        self.lock = threading.Lock()
        self._last_penalty = -1e9

    def _expire(self, now):
        while self.events and now - self.events[0][0] >= 60.0:
            self.spent -= self.events.popleft()[1]

    def acquire(self, tokens):
        tokens = min(tokens, self.budget)
        while True:
            with self.lock:
                now = time.monotonic()
                self._expire(now)
                if self.spent + tokens <= self.budget:
                    self.events.append((now, tokens)); self.spent += tokens
                    return
                sleep_for = min(60.0, 60.0 - (now - self.events[0][0])) + 0.01
            time.sleep(max(0.01, sleep_for))

    def penalise(self, seconds):
        with self.lock:
            now = time.monotonic()
            self._expire(now)
            if now - self._last_penalty < max(0.5, seconds):
                return
            self._last_penalty = now
            share = int(self.budget * min(1.0, max(0.0, seconds) / 60.0))
            if share:
                self.events.append((now, share)); self.spent += share


_RETRY_AFTER_RE = re.compile(r"try again in ([0-9.]+)\s*(ms|s)\b", re.I)

def _retry_delay(exc, attempt):
    hint = _RETRY_AFTER_RE.search(str(exc))
    if hint:
        v = float(hint.group(1))
        return (v / 1000.0 if hint.group(2).lower() == "ms" else v) + 0.25
    return min(30.0, 2 ** attempt) + random.random()


def judge_all():
    judged = load_judged()
    all_recs = [r for r in iter_generations() if r["question_id"] in _scope_qids()]
    pending = [r for r in all_recs if judge_key(r) not in judged and r.get("hypothesis")]
    missing_hyp = [r for r in all_recs if not r.get("hypothesis")]
    print(f"  {len(all_recs):,} generations in scope | {len(judged):,} already judged | "
          f"{len(pending):,} pending | {len(missing_hyp):,} have an empty hypothesis (unjudgeable)")
    if not pending:
        print("  nothing pending -- everything in scope is already judged." if judged else
              "  nothing pending and nothing judged: check that runs/ has generations.")
        return judged

    in_tok, usd = estimate_judge_cost(pending)
    print(f"  estimate: {len(pending):,} calls, ~{in_tok/1e6:.2f}M input tokens, ~${usd:.2f}")
    mins = in_tok / max(1.0, JUDGE_TPM * JUDGE_TPM_HEADROOM)
    print(f"  paced at {JUDGE_TPM:,} TPM x {JUDGE_TPM_HEADROOM:.0%} headroom -> expect roughly {mins:.0f} min")
    if not CONFIRM_JUDGE_SPEND:
        print("  CONFIRM_JUDGE_SPEND is False -- NOTHING WAS SENT and nothing was charged.")
        return judged

    pacer = TokenPacer(JUDGE_TPM, JUDGE_TPM_HEADROOM)
    write_lock = threading.Lock()

    def one(rec):
        last = None
        for attempt in range(JUDGE_MAX_ATTEMPTS):
            pacer.acquire(judge_tokens(rec))
            try:
                r = openai_client.chat.completions.create(**build_judge_request(rec))
                return judge_key(rec), "yes" in (r.choices[0].message.content or "").lower(), None
            except Exception as e:
                last = e
                delay = _retry_delay(e, attempt)
                if "rate limit" in str(e).lower() or "429" in str(e):
                    pacer.penalise(delay)
                time.sleep(delay)
        return judge_key(rec), None, last

    def run_pass(work, desc):
        from tqdm.auto import tqdm
        failed = []
        with open(JUDGED_PATH, "a", encoding="utf-8") as out:
            with ThreadPoolExecutor(max_workers=JUDGE_CONCURRENCY) as pool:
                for k, label, err in tqdm(pool.map(one, work), total=len(work), desc=desc):
                    if err is not None:
                        failed.append(k); continue
                    judged[k] = label
                    with write_lock:
                        out.write(json.dumps({"key": k, "label": label}) + "\n")
            out.flush()
        return failed

    failed_keys = set(run_pass(pending, "judging"))
    if failed_keys:
        retry = [r for r in pending if judge_key(r) in failed_keys]
        print(f"  {len(retry):,} rows did not land on the first pass -- retrying once more")
        time.sleep(15)
        still_failed = set(run_pass(retry, "judging (retry)"))
        if still_failed:
            print(f"  {len(still_failed):,} rows failed twice -- excluded from accuracy denominators.")
    return judged


def probe_judge_tpm():
    global JUDGE_TPM
    try:
        resp = openai_client.chat.completions.with_raw_response.create(
            model=JUDGE_MODEL, messages=[{"role": "user", "content": "Reply with: OK"}],
            temperature=0, max_tokens=5)
        limit = resp.headers.get("x-ratelimit-limit-tokens")
        remaining = resp.headers.get("x-ratelimit-remaining-tokens")
        if limit:
            JUDGE_TPM = int(limit)
            print(f"  probed real rate limit: {JUDGE_TPM:,} TPM (remaining: {remaining})")
        else:
            print(f"  rate-limit headers not present -- keeping JUDGE_TPM={JUDGE_TPM:,}")
    except Exception as e:
        print(f"  [warn] rate-limit probe failed ({e}) -- keeping JUDGE_TPM={JUDGE_TPM:,}")
    return JUDGE_TPM

probe_judge_tpm()

print("\nGate E6: priced dry-run (nothing sent, nothing charged) --\n")
_base_confirm = CONFIRM_JUDGE_SPEND
CONFIRM_JUDGE_SPEND = False
try:
    judge_all()
finally:
    CONFIRM_JUDGE_SPEND = _base_confirm

In [ ]:
# The real judging pass -- flip CONFIRM_JUDGE_SPEND above to False and re-run the dry-run cell to
# keep checking the estimate only.
JUDGED = judge_all()

## Section 18: Statistics

In [ ]:
from math import comb, sqrt

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, centre - half), min(1.0, centre + half)

def mcnemar_exact(a_labels, b_labels):
    b = sum(1 for x, y in zip(a_labels, b_labels) if x and not y)
    c = sum(1 for x, y in zip(a_labels, b_labels) if y and not x)
    n = b + c
    if n == 0:
        return b, c, 1.0
    k = min(b, c)
    tail = sum(comb(n, i) for i in range(0, k + 1)) / (2 ** n)
    return b, c, min(1.0, 2 * tail)

def arm_labels(reader, arm_name, qids=None):
    # reader is a required, explicit parameter -- unlike longmemeval_memorag.ipynb's own
    # arm_labels(), which hardcodes PRIMARY_READER (a real wart that notebook has, harmless there
    # since it only ever had one reader; fixed here from the start since this notebook has two).
    qids = qids or QUESTION_IDS
    out = {}
    p = RUNS / reader / f"{arm_name}.jsonl"
    if not p.exists():
        return out
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        k = judge_key(r)
        if k in JUDGED:
            out[r["question_id"]] = JUDGED[k]
    return {q: out[q] for q in qids if q in out}

def accuracy(reader, arm_name, qids=None):
    lab = arm_labels(reader, arm_name, qids)
    if not lab:
        return None, 0, 0
    k = sum(1 for v in lab.values() if v)
    return k / len(lab), k, len(lab)

def task_averaged(reader, arm_name, qids=None):
    lab = arm_labels(reader, arm_name, qids)
    if not lab:
        return None
    by = {}
    for q, v in lab.items():
        by.setdefault(QID2S[q]["question_type"], []).append(1 if v else 0)
    return float(np.mean([np.mean(v) for v in by.values()]))

print("statistics ready: wilson(), mcnemar_exact(), accuracy(reader, arm), task_averaged()")

## Section 19: Results -- headline figures, RWKV scaling, and the comparison against
`longmemeval_recreation.ipynb`

Loads `longmemeval_recreation.ipynb`'s own measured run from `/content/drive/MyDrive/lme_retry/`
(read-only). The two notebooks sampled their 100-question subsets independently (deviation #11),
so every paired statistic below is computed on the intersection, with the paired `n` printed;
each notebook's own full-n number is also shown, unpaired, for context.

In [ ]:
import matplotlib.pyplot as plt

COLOR_MEMORAG = "#1f77b4"   # memorag_top{k} -- retrieval steered by RWKV clues
COLOR_RAGQ    = "#ff7f0e"   # rag_q_top{k}   -- retrieval steered by the question alone
COLOR_MEMONLY = "#9467bd"   # memonly        -- NO retrieval; RWKV's own recall is the answer
COLOR_BASE    = "#888888"   # closedbook / oracle_con / s_con -- non-retrieval reference points
COLOR_REF     = "#333333"

def _bar_color(name):
    if "memonly" in name:
        return COLOR_MEMONLY
    if "memorag" in name:
        return COLOR_MEMORAG
    if "rag_q" in name:
        return COLOR_RAGQ
    return COLOR_BASE


# HEADLINE_RWKV_TAG picks which RWKV size represents MemoRAG/memonly on the one-glance headline
# chart below -- "g1i-2.9b" so it stays comparable to longmemeval_memorag.ipynb's own Figure 1,
# which only ever had the 2.9b model. The full size ladder (1.5b/2.9b/7.2b, top5 AND top10,
# memonly AND memorag) is broken out in Figure 1b and Figure 2 below -- nothing is hidden, it is
# just kept off the one chart meant to be read at a glance.
HEADLINE_RWKV_TAG = "g1i-2.9b"


def report_fig1_headline():
    print("=" * 100)
    print("Figure 1: headline accuracy ladder, per reader")
    print("  grey   = no retrieval (closedbook / oracle_con / s_con full-context)")
    print("  orange = retrieval, question-only query (rag_q)")
    print("  blue   = retrieval, RWKV-clue-steered query (memorag)")
    print("  purple = NO retrieval -- the reader sees only RWKV's own recall (memonly). This is")
    print("           the MemoRAG-side analogue of s_con: s_con hands the reader the raw text,")
    print("           memonly hands the reader RWKV's compressed memory of that same text instead.")
    print("=" * 100)
    core_arms = ([n for n, a in ARMS.items() if a.artifact == "core"]
                 + [f"memorag_top10__{HEADLINE_RWKV_TAG}", f"memonly__{HEADLINE_RWKV_TAG}"])
    dfs = []
    for reader in READER_ORDER:
        rows = []
        for name in core_arms:
            acc, k, n = accuracy(reader, name)
            if acc is None:
                continue
            _, lo, hi = wilson(k, n)
            rows.append(dict(reader=reader, arm=name, acc=acc, lo=lo, hi=hi, n=n))
        df = pd.DataFrame(rows)
        if df.empty:
            print(f"  {reader}: no judged results yet")
            continue
        df = df.sort_values("acc", ascending=True).reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(8, 0.45 * len(df) + 1.5))
        colors = [_bar_color(n) for n in df.arm]
        y = np.arange(len(df))
        ax.barh(y, df.acc, xerr=[df.acc - df.lo, df.hi - df.acc], color=colors,
               capsize=3, edgecolor="white", height=0.65)
        for yi, (acc, n) in enumerate(zip(df.acc, df.n)):
            ax.text(acc + 0.015, yi, f"{acc:.3f} (n={n})", va="center", fontsize=8)
        labels = [a.replace(f"__{HEADLINE_RWKV_TAG}", f" [RWKV {HEADLINE_RWKV_TAG}]") for a in df.arm]
        ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlim(0, min(1.0, df.hi.max() + 0.15))
        ax.set_xlabel("QA accuracy (Wilson 95% CI)")
        ax.set_title(f"Figure 1: accuracy ladder, {reader}, n={df.n.max()}", fontsize=11)
        for arm_name, style, label in [("oracle_con", "--", "oracle ceiling"),
                                        ("closedbook", ":", "closedbook floor")]:
            acc, _, _ = accuracy(reader, arm_name)
            if acc is not None:
                ax.axvline(acc, color=COLOR_REF, linestyle=style, linewidth=1, alpha=0.7)
                ax.text(acc, len(df) - 0.3, label, rotation=90, fontsize=7, va="top", ha="right", color=COLOR_REF)
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS / f"fig1_headline_{reader}.png", dpi=200); plt.show()
        df.to_csv(RESULTS / f"fig1_headline_{reader}.csv", index=False)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

report_fig1_headline()


In [ ]:
def report_fig1b_rnn_ladder():
    print("=" * 100)
    print("Figure 1b: every RNN-dependent arm, every RWKV size -- the detail behind Figure 1's")
    print("headline bars. Answers directly: does a bigger/smaller RWKV memory model change either")
    print("memonly (no retrieval, RWKV recall only) or memorag_top{5,10} (RWKV-steered retrieval)?")
    print("=" * 100)
    rnn_arms = [n for n, a in ARMS.items() if a.artifact == "rnn" and not n.startswith("memorag_trunc")]
    dfs = []
    for reader in READER_ORDER:
        rows = []
        for name in rnn_arms:
            acc, k, n = accuracy(reader, name)
            if acc is None:
                continue
            _, lo, hi = wilson(k, n)
            rows.append(dict(reader=reader, arm=name, acc=acc, lo=lo, hi=hi, n=n))
        df = pd.DataFrame(rows)
        if df.empty:
            print(f"  {reader}: no judged RNN-ladder results yet")
            continue
        df = df.sort_values("acc", ascending=True).reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(9, 0.4 * len(df) + 1.5))
        colors = [_bar_color(n) for n in df.arm]
        y = np.arange(len(df))
        ax.barh(y, df.acc, xerr=[df.acc - df.lo, df.hi - df.acc], color=colors,
               capsize=3, edgecolor="white", height=0.65)
        for yi, (acc, n) in enumerate(zip(df.acc, df.n)):
            ax.text(acc + 0.015, yi, f"{acc:.3f}", va="center", fontsize=8)

        def _label(a):
            kind, tag = a.rsplit("__", 1)
            params = f"{RWKV_PARAMS_B[tag]:g}B"
            kind = kind.replace("_top", " top")
            return f"{kind}, RWKV {params}"

        ax.set_yticks(y); ax.set_yticklabels([_label(a) for a in df.arm], fontsize=8)
        ax.set_xlim(0, min(1.0, df.hi.max() + 0.15))
        ax.set_xlabel("QA accuracy (Wilson 95% CI)")
        ax.set_title(f"Figure 1b: RNN-ladder detail, {reader}, n={df.n.max()}", fontsize=11)
        rq_acc, _, _ = accuracy(reader, "rag_q_top10")
        if rq_acc is not None:
            ax.axvline(rq_acc, color=COLOR_RAGQ, linestyle="--", linewidth=1, alpha=0.7)
            ax.text(rq_acc, len(df) - 0.3, "rag_q_top10\n(question-only control)", rotation=90,
                   fontsize=6, va="top", ha="right", color=COLOR_RAGQ)
        sc_acc, _, _ = accuracy(reader, "s_con")
        if sc_acc is not None:
            ax.axvline(sc_acc, color=COLOR_BASE, linestyle=":", linewidth=1, alpha=0.7)
            ax.text(sc_acc, len(df) - 0.3, "s_con\n(full context, no retrieval)", rotation=90,
                   fontsize=6, va="top", ha="right", color=COLOR_BASE)
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS / f"fig1b_rnn_ladder_{reader}.png", dpi=200); plt.show()
        df.to_csv(RESULTS / f"fig1b_rnn_ladder_{reader}.csv", index=False)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

report_fig1b_rnn_ladder()


In [ ]:
def report_fig2_rnn_scaling():
    print("=" * 100)
    print("Figure 2: does a bigger RWKV memory model produce better retrieval clues or better")
    print("standalone recall?")
    print("  o memorag_top10 / ^ memorag_top5 = retrieval steered by RWKV clues, two evidence budgets")
    print("  s memonly                        = no retrieval; the reader sees only RWKV's recall")
    print("  dashed grey line = rag_q_top10, a question-only retrieval control at the SAME budget")
    print("=" * 100)
    rows = []
    for reader in READER_ORDER:
        for tag in RWKV_ACTIVE:
            for kind, arm_fmt in [("memorag_top10", "memorag_top10__{}"),
                                  ("memorag_top5", "memorag_top5__{}"),
                                  ("memonly", "memonly__{}")]:
                acc, k, n = accuracy(reader, arm_fmt.format(tag))
                if acc is None:
                    continue
                _, lo, hi = wilson(k, n)
                rows.append(dict(reader=reader, kind=kind, tag=tag, params_b=RWKV_PARAMS_B[tag],
                                 acc=acc, lo=lo, hi=hi, n=n))
    df = pd.DataFrame(rows)
    if df.empty:
        print("  no judged results yet"); return df

    fig, axes = plt.subplots(1, len(READER_ORDER), figsize=(6 * len(READER_ORDER), 4), sharey=True)
    if len(READER_ORDER) == 1:
        axes = [axes]
    kinds = [("memorag_top10", COLOR_MEMORAG, "o", "-"), ("memorag_top5", COLOR_MEMORAG, "^", "--"),
            ("memonly", COLOR_MEMONLY, "s", "-")]
    for ax, reader in zip(axes, READER_ORDER):
        for kind, color, marker, ls in kinds:
            sub = df[(df.reader == reader) & (df.kind == kind)].sort_values("params_b")
            if sub.empty:
                continue
            ax.errorbar(sub.params_b, sub.acc, yerr=[sub.acc - sub.lo, sub.hi - sub.acc],
                        marker=marker, linestyle=ls, color=color, label=kind, capsize=3)
            for _, r in sub.iterrows():
                ax.annotate(f"{r.acc:.2f}", (r.params_b, r.acc), textcoords="offset points",
                           xytext=(0, 8), fontsize=7, ha="center")
        ragq_acc, _, _ = accuracy(reader, "rag_q_top10")
        if ragq_acc is not None:
            ax.axhline(ragq_acc, color=COLOR_RAGQ, linestyle="--", linewidth=1,
                       label="rag_q_top10 (question-only control)")
        ax.set_xscale("log")
        ax.set_xticks([RWKV_PARAMS_B[t] for t in RWKV_ACTIVE])
        ax.set_xticklabels([f"{RWKV_PARAMS_B[t]}B" for t in RWKV_ACTIVE])
        ax.set_xlabel("RWKV-7 memory model size"); ax.set_title(reader, fontsize=10)
        ax.grid(alpha=0.3); ax.legend(fontsize=7)
    axes[0].set_ylabel("QA accuracy (Wilson 95% CI)")
    fig.suptitle("Figure 2: accuracy vs. RWKV memory-model size", fontsize=12)
    plt.tight_layout(); plt.savefig(RESULTS / "fig2_rnn_scaling.png", dpi=200); plt.show()
    df.to_csv(RESULTS / "fig2_rnn_scaling.csv", index=False)
    print("\n  A flat curve here is an expected, legitimate result -- new_method.ipynb measured "
          "plain-question BM25 already hitting an oracle session in its top-10 for 20/20 sample "
          "questions, leaving little headroom at k=10 over a ~47-session haystack for any query "
          "source, bigger memory model included, to improve on.")
    return df

report_fig2_rnn_scaling()


In [ ]:
def recreation_labels(reader, arm_name):
    judged_path = RECREATION_REPO / "results" / "judged.jsonl"
    run_path = RECREATION_REPO / "runs" / reader / f"{arm_name}.jsonl"
    if not judged_path.exists() or not run_path.exists():
        return {}
    judged = {}
    for line in judged_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line); judged[r["key"]] = r["label"]
    out = {}
    for line in run_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            k = f"{reader}|{arm_name}|{r['question_id']}"
            if k in judged:
                out[r["question_id"]] = judged[k]
    return out

def recreation_available():
    return (RECREATION_REPO / "results" / "judged.jsonl").exists()

def recreation_question_ids():
    # longmemeval_recreation.ipynb's own choose_questions() never writes a question_subset_*.json
    # cache file -- it only *reads* an optional prior-run reference file and otherwise samples in
    # memory. So there was nothing under RECREATION_REPO/store to glob for, which is why
    # PAIRED_QIDS was silently empty before this fix. Recover the actual 100 question IDs the way
    # every other piece of this notebook already does: from its own persisted generation records,
    # which carry question_id per row and exist for any reader judged at all (oracle_con / s_con
    # are judged for every reader -- Fig3b+Tab8 scope = None in that notebook's Section 16).
    for reader in READER_ORDER:
        for arm in ("oracle_con", "s_con"):
            p = RECREATION_REPO / "runs" / reader / f"{arm}.jsonl"
            if p.exists():
                ids = {json.loads(l)["question_id"] for l in p.read_text(encoding="utf-8").splitlines() if l.strip()}
                if ids:
                    return ids
    return set()

PAIRED_QIDS = sorted(set(QUESTION_IDS) & recreation_question_ids()) if recreation_available() else []
print(f"Paired question intersection with longmemeval_recreation.ipynb: {len(PAIRED_QIDS)} of "
      f"{len(QUESTION_IDS)} (recreation subset size {len(recreation_question_ids())})")


# Table 8's memory-design arms (tab8_round_V / tab8_round_V_fact) were judged in
# longmemeval_recreation.ipynb for llama-3.1-8b-instruct ONLY (its own Section 16 JUDGE_SCOPE, a
# deliberate cost-saving decision documented there) -- a NaN for other readers below means "not
# judged upstream", not missing/broken data here. s_con and oracle_con have no such gap
# (Fig3b+Tab8 scope = every reader).
RECREATION_ARM_SCOPE_NOTE = (
    "tab8_round_V (recreation's RAG K=V arm) was only judged for llama-3.1-8b-instruct in "
    "longmemeval_recreation.ipynb -- a NaN for other readers reflects that upstream scoping "
    "decision, not missing/broken data here.")


def report_sanity_vs_recreation():
    print("=" * 100)
    print("Sanity check AND headline comparison: this notebook's arms vs. longmemeval_recreation")
    print(".ipynb's ACTUAL measured run (not the paper's published numbers). Includes the MemoRAG")
    print("arms -- this is the table that answers 'does MemoRAG beat the plain baseline?'")
    print("=" * 100)
    if not recreation_available():
        print(f"  {RECREATION_REPO} not found or has no judged.jsonl -- run "
              f"longmemeval_recreation.ipynb at least through its judging section first.")
        return None
    rows = []
    for reader in READER_ORDER:
        comparisons = [("closedbook", "closedbook"), ("oracle_con", "oracle_con"),
                      ("s_con", "s_con"), ("rag_q_top10", "tab8_round_V"),
                      (f"memonly__{HEADLINE_RWKV_TAG}", None),
                      (f"memorag_top10__{HEADLINE_RWKV_TAG}", None)]
        for arm, recreation_arm in comparisons:
            this_acc, this_k, this_n = accuracy(reader, arm)
            if recreation_arm is None:
                rec_acc, rec_n = None, 0
            else:
                rec_lab = recreation_labels(reader, recreation_arm)
                rec_acc = (sum(rec_lab.values()) / len(rec_lab)) if rec_lab else None
                rec_n = len(rec_lab)
            rows.append(dict(reader=reader, arm=arm, this_notebook=this_acc, this_n=this_n,
                             recreation_actual=rec_acc, recreation_n=rec_n))
    df = pd.DataFrame(rows)
    print(df.round(3).to_string(index=False))
    print(f"\n  {RECREATION_ARM_SCOPE_NOTE}")
    print("  memonly/memorag rows have no recreation-notebook counterpart by design (that "
          "notebook never ran MemoRAG) -- they are included here so the MemoRAG-vs-baseline "
          "comparison lives in one table instead of being scattered across figures. Compare "
          "the memonly/memorag_top10 rows against THIS NOTEBOOK's own rag_q_top10/s_con rows "
          "directly above them, not against their (blank) recreation_actual column.")
    print("  (different, independently-sampled 100-question subsets -- rough agreement is "
          "expected, not exact. See the paired McNemar test in Figure 3 for a statistic computed "
          "on the actual question intersection.)")
    df.to_csv(RESULTS / "sanity_vs_recreation.csv", index=False)
    return df

report_sanity_vs_recreation()


In [ ]:
def report_fig3_three_way():
    print("=" * 100)
    print("Figure 3: recreation baseline vs. MemoRAG, per reader, at each RWKV size")
    print("(paired McNemar test on the ACTUAL intersected question set -- printed n, not assumed)")
    print("=" * 100)
    if not PAIRED_QIDS:
        print("  no paired question intersection available (recreation subset not found) -- skipping")
        return None

    dfs = []
    for reader in READER_ORDER:
        # Prefer recreation's own RAG arm (tab8_round_V) as the baseline anchor -- the more
        # apples-to-apples "retrieval-augmented" comparison for MemoRAG. It was only judged for
        # llama-3.1-8b-instruct upstream (RECREATION_ARM_SCOPE_NOTE), so readers without it fall
        # back to s_con (full-context, no retrieval -- judged for every reader), labelled
        # accordingly rather than silently substituted.
        anchor_arm, anchor_label = "tab8_round_V", "recreation baseline (RAG K=V)"
        rec_lab_full = recreation_labels(reader, anchor_arm)
        if not rec_lab_full:
            anchor_arm, anchor_label = "s_con", "recreation baseline (S, full context)"
            rec_lab_full = recreation_labels(reader, anchor_arm)
        rec_lab = {q: v for q, v in rec_lab_full.items() if q in PAIRED_QIDS}

        rows = []
        if rec_lab:
            k = sum(1 for v in rec_lab.values() if v)
            _, lo, hi = wilson(k, len(rec_lab))
            rows.append(dict(system=anchor_label, acc=k / len(rec_lab), lo=lo, hi=hi, n=len(rec_lab)))
        else:
            print(f"  {reader}: no recreation judgements available on the paired set for either "
                  f"tab8_round_V or s_con -- skipping this reader")
            continue

        for tag in RWKV_ACTIVE:
            for kind in ("memonly", "memorag_top5", "memorag_top10"):
                lab = {q: v for q, v in arm_labels(reader, f"{kind}__{tag}").items() if q in PAIRED_QIDS}
                if lab:
                    k = sum(1 for v in lab.values() if v)
                    _, lo, hi = wilson(k, len(lab))
                    rows.append(dict(system=f"{kind}, RWKV {RWKV_PARAMS_B[tag]:g}B",
                                     acc=k / len(lab), lo=lo, hi=hi, n=len(lab)))
        df = pd.DataFrame(rows)
        if df.empty or len(df) == 1:
            print(f"  {reader}: nothing judged yet on the paired question set"); continue

        fig, ax = plt.subplots(figsize=(8, 0.5 * len(df) + 1.5))
        y = np.arange(len(df))
        colors = [COLOR_BASE if s == anchor_label else _bar_color(s) for s in df.system]
        ax.barh(y, df.acc, xerr=[df.acc - df.lo, df.hi - df.acc], color=colors, capsize=3)
        for yi, (acc, n) in enumerate(zip(df.acc, df.n)):
            ax.text(acc + 0.015, yi, f"{acc:.3f} (n={n})", va="center", fontsize=8)
        ax.set_yticks(y); ax.set_yticklabels(df.system, fontsize=9)
        ax.set_xlim(0, min(1.0, df.hi.max() + 0.15))
        ax.set_xlabel(f"QA accuracy, paired n={len(PAIRED_QIDS)}")
        ax.set_title(f"Figure 3: recreation baseline vs. MemoRAG ladder, {reader}", fontsize=11)
        ax.grid(axis="x", alpha=0.3)
        plt.tight_layout(); plt.savefig(RESULTS / f"fig3_three_way_{reader}.png", dpi=200); plt.show()
        df.to_csv(RESULTS / f"fig3_three_way_{reader}.csv", index=False)

        print(f"\n  {reader}: paired McNemar vs. {anchor_label} (n={len(rec_lab)} on the "
              f"intersected set):")
        for tag in RWKV_ACTIVE:
            memo_lab = {q: v for q, v in arm_labels(reader, f"memorag_top10__{tag}").items() if q in PAIRED_QIDS}
            common = sorted(set(rec_lab) & set(memo_lab))
            if len(common) < 5:
                print(f"    memorag_top10@{tag}: fewer than 5 paired questions ({len(common)}) -- skipped")
                continue
            a = [rec_lab[q] for q in common]
            b = [memo_lab[q] for q in common]
            bdis, cdis, p = mcnemar_exact(a, b)
            sig = "(significant at .05)" if p < 0.05 else "(not significant at .05)"
            print(f"    memorag_top10@{tag}: paired n={len(common)}  b={bdis} c={cdis}  p={p:.3f}  {sig}")
        dfs.append(df.assign(reader=reader))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

report_fig3_three_way()


In [ ]:
def cost_ledger():
    print("=" * 90)
    print("Cost and volume ledger (measured from the run records, not estimated)")
    print("=" * 90)
    agg = {}
    for r in iter_generations():
        k = (r.get("reader", "?"), r.get("artifact", "?"))
        a = agg.setdefault(k, dict(n=0, pin=0, pout=0, cost=0.0, lat=0.0, err=0))
        a["n"] += 1; a["pin"] += r.get("prompt_tokens", 0) or 0; a["pout"] += r.get("completion_tokens", 0) or 0
        a["cost"] += r.get("cost_usd", 0.0) or 0.0; a["lat"] += r.get("latency_s", 0.0) or 0.0
        a["err"] += 1 if r.get("error") else 0
    if not agg:
        print("  no generations on disk yet"); return None
    rows = [dict(reader=k[0], artifact=k[1], n=v["n"], in_Mtok=v["pin"] / 1e6, out_Mtok=v["pout"] / 1e6,
                gen_cost_usd=v["cost"], mean_latency_s=v["lat"] / max(1, v["n"]), errors=v["err"])
           for k, v in sorted(agg.items())]
    df = pd.DataFrame(rows)
    print(df.round(3).to_string(index=False))

    n_judged = len(JUDGED) if "JUDGED" in globals() else 0
    judge_cost = n_judged * 400 / 1e6 * PRICE_IN_PER_MTOK   # same simplified pricing convention
                                                              # longmemeval_memorag.ipynb's own
                                                              # cost_ledger() uses.
    gen_cost = df.gen_cost_usd.sum()
    rwkv_rows = sum(len(MEMORY.get(t, {})) for t in RWKV_ACTIVE) if CAN_RUN_RWKV else 0
    print(f"\n  generation (OpenRouter + local vLLM):  ${gen_cost:8.2f}")
    print(f"  RWKV memory-building:                  local, GPU time, not billed ({rwkv_rows} rows "
          f"across {len(RWKV_ACTIVE)} tags)")
    print(f"  judge ({n_judged:,} calls, GPT-4o):          ${judge_cost:8.2f}")
    print(f"  {'-' * 52}")
    print(f"  TOTAL API SPEND SO FAR:                ${gen_cost + judge_cost:8.2f}")
    df.to_csv(RESULTS / "cost_ledger.csv", index=False)

    print("\nResults written to:", RESULTS)
    for p in sorted(RESULTS.glob("*")):
        print(" ", p.name)
    return df

cost_ledger()